# 09 เก็บข้อมูลภายนอกจริงและหลักฐานวันที่ (v2 – งานวิจัยระหว่างดำเนินการ)

**เป้าหมาย:** (1) ตรวจหลักฐานรูปแบบวันที่รายปีจาก XLSX ทางการของ MOT ที่มีเซลล์วันที่แบบ typed โดยจับคู่ด้วย ACC_CODE เดิมแบบตรงตัวเท่านั้น (2) เก็บ/ตรวจแหล่งข้อมูลภายนอกจริงสำหรับ v2: ปริมาณจราจร AADT ของกรมทางหลวง, ขอบเขตจังหวัดและจุดสอบถามสภาพอากาศ, พยากรณ์อากาศย้อนหลังที่เก็บถาวร (JMA GSM, ล่วงหน้า 48 ชม.), ปฏิทินวันหยุดธนาคารแห่งประเทศไทย (3) ทำตารางความพร้อมของ feature ทุกกลุ่ม (เทศกาล, อากาศ, ฤดูกาล, ช่วงเวลา, จราจร, ประวัติ)

**ไม่ทำในขั้นนี้:** ไม่สร้าง feature panel, ไม่ fit/เทียบโมเดล, ไม่อ้างว่าชนะ baseline, ไม่แก้ข้อมูล/รายงาน/notebook/config ของ v1 (01–08)

**อินพุต:** `config/external_sources_v2.json`; snapshot ดิบ v1 `data/raw/accidents/*.csv` (อ่านอย่างเดียว ตรวจ SHA256 กับ `source_manifest.json`); ไฟล์ preflight ที่ Codex ดาวน์โหลดไว้ใน `data/external_v2/evidence/` (อ่านอย่างเดียว บันทึก hash และ mtime โดยไม่อ้างเวลาดาวน์โหลดเดิม); การดาวน์โหลด HTTPS จริงจากโฮสต์ใน allowlist (ไม่มี credential ไม่มีบริการเสียเงิน)

**เอาต์พุต:** ข้อมูลดิบ/แคช/ตารางรายแถว (ไม่อยู่ใน git) ใต้ `ROADAWARE_V2_EXTERNAL_ROOT` (ค่าเริ่มต้น `backend/data/external_v2/`: `registry.json`, `raw/…`, `derived/…`); รายงานขนาดเล็กใต้ `ROADAWARE_V2_OUTPUT_ROOT/reports/v2/sources/`: `sources_manifest.json`, `date_evidence_by_year.csv/json`, `external_schema_coverage.csv`, `feature_source_readiness.csv`, `traffic_resource_metadata.csv`, `province_points.csv`, `weather_coverage.csv`, `calendar_records.csv`, `source_audit_th.md`

**การตรวจ:** แคชใช้ซ้ำได้เฉพาะเมื่อ SHA256 ตรงกับ registry (ไม่ตรง = หยุดชัดเจน ไม่เขียนทับ); แหล่งที่ล้มเหลวถูกบันทึกเป็น unavailable ไม่ปลอมข้อมูล; วันที่ resolve ได้เฉพาะแถวที่ ID ตรงและวันที่ typed เห็นพ้องกันโดยไม่กำกวม; อากาศรวมรายวัน/6 ชม. ต้องครบชั่วโมง (ไม่รวมค่าว่างเป็น 0) และผ่านเงื่อนไขเวลาออกพยากรณ์; จุดจังหวัดต้องอยู่ในรูปจังหวัด; JSON เป็น strict (ไม่มี NaN)

**ข้อจำกัด:** notebook นี้รันหลายนาที (อากาศ 77 จังหวัด × 5 ปี แบ่ง chunk และหยุดเมื่อโดน rate limit แล้วรันต่อได้); AADT เป็นค่าเฉลี่ยรายปี ณ จุดสำรวจ ไม่ใช่จราจร real-time และรวมกันเป็นจำนวนรถไม่ได้; เวลาใน catalog เป็นเพียง proxy ของวันที่เผยแพร่; ระยะ lead ของ API ไม่ได้พิสูจน์ความหน่วงการเผยแพร่จริงในอดีต; วันหยุด ธปท. ไม่ใช่วันหยุดราชการทั้งหมด; ขอบเขต geoBoundaries มาจาก OSM ไม่ใช่ข้อมูลทางการของรัฐ

**รันอย่างไร:** VS Code เลือก kernel `.venv` แล้ว *Run All* (เปิดจาก root หรือโฟลเดอร์ notebooks ก็ได้) หรือ headless: `.venv/bin/python backend/scripts/run_v2_notebooks.py --stage collect`

In [1]:
# FUNCTIONS (notebook 09) - self-contained; tests execute this cell directly (no network in tests)
import datetime as dt, hashlib, html as htmllib, io, json, math, os, re, socket, time, zipfile
from pathlib import Path
from urllib.parse import urlencode, urljoin, urlparse
from urllib.request import HTTPRedirectHandler, Request, build_opener
from urllib.error import HTTPError, URLError
import numpy as np, pandas as pd

EXCEL_ORIGIN = dt.date(1899, 12, 30)
TYPED_KINDS = ('typed_datetime', 'typed_date')
THAI_MONTHS = {'มกราคม': 1, 'กุมภาพันธ์': 2, 'มีนาคม': 3, 'เมษายน': 4, 'พฤษภาคม': 5, 'มิถุนายน': 6,
               'กรกฎาคม': 7, 'สิงหาคม': 8, 'กันยายน': 9, 'ตุลาคม': 10, 'พฤศจิกายน': 11, 'ธันวาคม': 12}
THAI_DIGITS = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
VERIFIED_FESTIVAL = 'verified_recurring_fixed_festival_in_official_source'
EXCLUDED_TEXT = 'excluded_unreviewed_text_extraction'
CAL_COLUMNS = ['date', 'name', 'festival_id', 'status', 'record_type', 'is_national_public_holiday', 'source_id', 'source_url',
               'source_sha256', 'source_year', 'announcement_date', 'announcement_date_status', 'source_text']
POINT_COLUMNS = ['province_code', 'name_th', 'name_en', 'shape_name', 'shape_iso', 'shape_id', 'shape_index', 'join_method', 'lat', 'lon',
                 'point_within_polygon', 'geometry_valid', 'in_v1_panel', 'point_method', 'point_meaning', 'geometry_source',
                 'geometry_sha256', 'province_table_sha256']


# ---------- generic I/O ----------
def utc_now():
    return dt.datetime.now(dt.timezone.utc).replace(microsecond=0).isoformat()


def sha256_bytes(b):
    return hashlib.sha256(b).hexdigest()


def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def json_safe(o):
    """Strict-JSON friendly copy: NaN/inf/NaT/NA -> None, numpy scalars -> Python, dates -> ISO strings."""
    if o is None or o is pd.NaT or o is pd.NA:
        return None
    if isinstance(o, dict):
        return {str(k): json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [json_safe(v) for v in o]
    if isinstance(o, (bool, np.bool_)):
        return bool(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, (float, np.floating)):
        return float(o) if math.isfinite(float(o)) else None
    if isinstance(o, (pd.Timestamp, dt.datetime, dt.date)):
        return o.isoformat()
    if isinstance(o, Path):
        return str(o)
    return o


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(json_safe(obj), indent=2, ensure_ascii=False, allow_nan=False), 'utf-8')
    os.replace(tmp, path)


def flatten_records(records):
    """List of dicts -> DataFrame; nested dict/list values become JSON strings (strict)."""
    rows = [{k: (json.dumps(json_safe(v), ensure_ascii=False, allow_nan=False) if isinstance(v, (dict, list, tuple)) else v)
             for k, v in r.items()} for r in records]
    return pd.DataFrame(rows)


def write_csv(path, df):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df = df.copy()
    for c in df.columns:
        if df[c].map(lambda v: isinstance(v, (dict, list, tuple))).any():
            df[c] = df[c].map(lambda v: json.dumps(json_safe(v), ensure_ascii=False) if isinstance(v, (dict, list, tuple)) else v)
    df.to_csv(path, index=False)
    return path


def write_table(base, df):
    """Parquet when pyarrow is installed, else deterministic gzip CSV (write_csv_gz: mtime 0). Returns the written path (str)."""
    base = Path(base)
    base.parent.mkdir(parents=True, exist_ok=True)
    try:
        import pyarrow  # noqa: F401
        p = base.with_name(base.name + '.parquet')
        df.to_parquet(p, index=False)
    except ImportError:
        p = base.with_name(base.name + '.csv.gz')
        write_csv_gz(p, df)
    return str(p)


def functions_cell_source(nb_path):
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    srcs = [''.join(c['source']) if isinstance(c['source'], list) else c['source'] for c in nb['cells'] if c['cell_type'] == 'code']
    hits = [s for s in srcs if s.lstrip().startswith('# FUNCTIONS')]
    if len(hits) != 1:
        raise ValueError(f'{nb_path}: expected exactly one FUNCTIONS cell, found {len(hits)}')
    return hits[0]


def verify_v1_snapshot(v1_root, manifest):
    """sha256 of every v1 raw snapshot file; raises when any differs from the v1 source_manifest (read-only check)."""
    out, bad = {}, []
    for f in manifest['files']:
        p = Path(v1_root) / f['path']
        sha = sha256_file(p) if p.is_file() else None
        if sha != f['sha256']:
            bad.append(f"{f['path']}: {sha} != {f['sha256']}")
        out[f['path']] = sha
    if bad:
        raise RuntimeError('v1 raw snapshot does not match its manifest (v1 must stay unchanged):\n' + '\n'.join(bad))
    return out


# ---------- network: allowlist, bounded retries, size cap, 429 handling ----------
class FetchError(Exception):
    def __init__(self, msg, code=None, retry_after=None):
        super().__init__(msg)
        self.code = code
        self.retry_after = retry_after


class RateLimited(FetchError):
    pass


class BudgetExhausted(FetchError):
    pass


class CacheIntegrityError(Exception):
    pass


class RequestBudget:
    """Counts network attempts (including retries); a child budget also charges its parent."""
    def __init__(self, max_requests, parent=None):
        self.max_requests = int(max_requests)
        self.used = 0
        self.parent = parent

    def take(self):
        if self.used >= self.max_requests:
            raise BudgetExhausted(f'network request budget {self.max_requests} exhausted')
        if self.parent is not None:
            self.parent.take()
        self.used += 1


def host_allowed(url, allowed_hosts):
    u = urlparse(str(url))
    return u.scheme == 'https' and (u.hostname or '').lower() in {h.lower() for h in allowed_hosts}


def make_opener(allowed_hosts):
    class _AllowlistRedirect(HTTPRedirectHandler):
        def redirect_request(self, req, fp, code, msg, headers, newurl):
            if not host_allowed(newurl, allowed_hosts):
                raise FetchError(f'redirect to non-allowlisted URL refused: {newurl}')
            return super().redirect_request(req, fp, code, msg, headers, newurl)
    op = build_opener(_AllowlistRedirect)
    return lambda req, timeout: op.open(req, timeout=timeout)


def parse_retry_after(value):
    if value is None:
        return None
    try:
        return max(0.0, float(str(value).strip()))
    except ValueError:
        return None  # HTTP-date form is treated as unknown -> stop rather than guess


def http_get(url, net, budget, opener=None, sleep=time.sleep, max_bytes=None, log=print):
    """GET with https+allowlist (also after redirects), bounded retries/backoff, response-size cap and 429 Retry-After."""
    if not host_allowed(url, net['allowed_hosts']):
        raise FetchError(f'URL not allowlisted (https and an allowed host required): {url}')
    opener = opener or make_opener(net['allowed_hosts'])
    cap = int(max_bytes or net['max_response_bytes'])
    retries = int(net['max_retries'])
    attempt = 0
    while True:
        budget.take()
        attempt += 1
        try:
            req = Request(url, headers={'User-Agent': net['user_agent'], 'Accept': '*/*'})
            with opener(req, float(net['timeout_seconds'])) as resp:
                final = resp.geturl()
                if not host_allowed(final, net['allowed_hosts']):
                    raise FetchError(f'resolved URL not allowlisted: {final}')
                body = resp.read(cap + 1)
                if len(body) > cap:
                    raise FetchError(f'response exceeds max_response_bytes={cap}: {url}')
                h = resp.headers
                return {'url': url, 'resolved_url': final, 'http_status': getattr(resp, 'status', None), 'attempts': attempt,
                        'retrieved_at_utc': utc_now(), 'body': body,
                        'headers': {k: h.get(k) for k in ('Content-Type', 'Content-Length', 'Last-Modified', 'ETag')}}
        except HTTPError as e:
            ra = parse_retry_after(e.headers.get('Retry-After') if e.headers is not None else None)
            try:
                snippet = e.read(500).decode('utf-8', errors='replace')
            except Exception:
                snippet = ''
            if e.code == 429:
                if ra is not None and ra <= float(net['max_retry_after_seconds']) and attempt <= retries:
                    log(f'    HTTP 429 for {url}; waiting Retry-After={ra}s')
                    sleep(ra)
                    continue
                raise RateLimited(f'HTTP 429 rate limited (Retry-After={ra}) for {url}; stopping instead of retrying. {snippet}',
                                  code=429, retry_after=ra)
            if e.code in (500, 502, 503, 504) and attempt <= retries:
                sleep(float(net['backoff_seconds']) * 2 ** (attempt - 1))
                continue
            raise FetchError(f'HTTP {e.code} for {url}. {snippet}'.strip(), code=e.code)
        except FetchError:
            raise
        except (URLError, socket.timeout, TimeoutError, ConnectionError, OSError) as e:
            if attempt <= retries:
                sleep(float(net['backoff_seconds']) * 2 ** (attempt - 1))
                continue
            raise FetchError(f'network error for {url} after {attempt} attempts: {type(e).__name__}: {e}')


# ---------- hash-verified cache registry ----------
def load_registry(path):
    p = Path(path)
    if p.is_file():
        return json.loads(p.read_text('utf-8'))
    return {'schema': 'roadaware-external-v2-registry/1', 'resources': {}}


def _write_bytes(path, body):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.part')
    tmp.write_bytes(body)
    os.replace(tmp, path)


def _free_version_path(root, rel, start):
    p = Path(rel)
    n = start
    while (Path(root) / p.with_name(f'{p.stem}.v{n}{p.suffix}')).exists():
        n += 1
    return p.with_name(f'{p.stem}.v{n}{p.suffix}').as_posix()


def cached_fetch(res_id, url, rel, root, registry, net, budget, opener=None, sleep=time.sleep, refresh=False, offline=False,
                 validate=None, meta=None, max_bytes=None, log=print):
    """Return the registry entry for res_id. A cached file is reused only when its sha256 matches the registry; a mismatch
    raises (file preserved, never overwritten). Re-downloaded bytes that differ from the registry become a new version file."""
    root = Path(root)
    res = registry.setdefault('resources', {})
    entry = res.get(res_id)
    has_reg = bool(entry and entry.get('sha256') and entry.get('path'))
    if has_reg and entry.get('url') and entry['url'] != url:
        raise CacheIntegrityError(f"{res_id}: registered URL {entry['url']} != requested {url}; a cached snapshot of a different query "
                                  'is never relabelled. Use a new resource id or move the old entry/file aside.')
    if has_reg and (root / entry['path']).is_file():
        sha = sha256_file(root / entry['path'])
        if sha != entry['sha256']:
            raise CacheIntegrityError(f"{res_id}: cached file {entry['path']} sha256 {sha[:12]} != registry {entry['sha256'][:12]}; "
                                      'file preserved (not overwritten). Restore it or move it aside, then rerun.')
        if not refresh:
            entry.update(last_status='cache_hit_hash_verified', verified_at_utc=utc_now())
            return entry
    elif not has_reg and (root / rel).is_file():
        raise CacheIntegrityError(f'{res_id}: unregistered file already at {rel}; refusing to adopt or overwrite it. Move it aside and rerun.')
    if offline:
        raise FetchError(f'{res_id}: offline mode and no hash-verified cached copy')
    r = http_get(url, net, budget, opener=opener, sleep=sleep, max_bytes=max_bytes, log=log)
    body = r.pop('body')
    if validate is not None:
        validate(body)
    sha = sha256_bytes(body)
    versions = list(entry.get('versions', [])) if entry else []
    if has_reg and sha == entry['sha256']:
        path, status = entry['path'], 'refetched_identical'
    elif has_reg:
        versions.append({k: entry.get(k) for k in ('path', 'sha256', 'bytes', 'retrieved_at_utc', 'verified_at_utc')})
        path, status = _free_version_path(root, entry['path'], len(versions) + 1), 'remote_bytes_changed_new_version'
    else:
        path, status = Path(rel).as_posix(), 'downloaded'
    if not (root / path).is_file():
        _write_bytes(root / path, body)
    new = {'resource_id': res_id, 'url': url, 'resolved_url': r['resolved_url'], 'path': path, 'sha256': sha, 'bytes': len(body),
           'retrieved_at_utc': r['retrieved_at_utc'], 'verified_at_utc': utc_now(), 'http_status': r['http_status'],
           'attempts': r['attempts'], 'headers': r['headers'], 'provenance': 'downloaded_by_notebook09', 'last_status': status,
           'versions': versions}
    new.update(meta or {})
    res[res_id] = new
    return new


def try_fetch(res_id, url, rel, root, registry, net, budget, required=False, **kw):
    """cached_fetch, but optional resources that fail are recorded as unavailable (never faked). Integrity errors always raise."""
    try:
        return cached_fetch(res_id, url, rel, root, registry, net, budget, **kw)
    except FetchError as e:
        if required:
            raise
        res = registry.setdefault('resources', {})
        e0 = dict(res.get(res_id) or {})
        e0.update(resource_id=res_id, url=url, last_status='unavailable', error=str(e)[:1000], error_type=type(e).__name__,
                  attempted_at_utc=utc_now())
        if not e0.get('sha256'):
            e0['path'] = None
        res[res_id] = e0
        print(f'    unavailable: {res_id}: {str(e)[:300]}')
        out = dict(e0)
        out['sha256'] = None  # this run has no usable bytes
        return out


def register_preflight(res_id, url, path, registry, provenance, meta=None):
    """Register a Codex preflight file (read-only): hash, size and cache mtime; original download time stays unknown.
    If the bytes differ from an earlier registration the old record is kept under versions and the change is flagged."""
    p = Path(path)
    sha = sha256_file(p)
    st = p.stat()
    res = registry.setdefault('resources', {})
    old = res.get(res_id)
    versions = list(old.get('versions', [])) if old else []
    status = 'preflight_hash_registered'
    if old and old.get('sha256'):
        if old['sha256'] == sha:
            status = 'preflight_hash_verified'
        else:
            raise CacheIntegrityError(f"{res_id}: preflight file {p} sha256 {sha[:12]} != registered {old['sha256'][:12]}; the registered "
                                      'bytes cannot be preserved after an external change, so they are not adopted. Restore the file '
                                      'or investigate and deregister it explicitly.')
    e = {'resource_id': res_id, 'url': url, 'preflight_path': str(p), 'path': None, 'sha256': sha, 'bytes': st.st_size,
         'cache_mtime_utc': dt.datetime.fromtimestamp(st.st_mtime, dt.timezone.utc).replace(microsecond=0).isoformat(),
         'verified_at_utc': utc_now(), 'retrieved_at_utc': None, 'original_download_time': 'unknown', 'provenance': provenance,
         'last_status': status, 'versions': versions}
    e.update(meta or {})
    res[res_id] = e
    return e


def json_validator(require_success=False):
    def v(body):
        try:
            o = json.loads(body.decode('utf-8'))
        except Exception as e:
            raise FetchError(f'invalid JSON response: {e}')
        if require_success and not (isinstance(o, dict) and o.get('success') is True):
            raise FetchError('CKAN response without success == true')
    return v


def nonempty_validator(body):
    if not body or not body.strip():
        raise FetchError('empty response body')


def xlsx_validator(body):
    if not zipfile.is_zipfile(io.BytesIO(body)):
        raise FetchError('response is not a complete XLSX (zip) file')


def pdf_validator(body):
    if not body.lstrip()[:5].startswith(b'%PDF'):
        raise FetchError('response is not a PDF')


def weather_validator(n_locations):
    def v(body):
        try:
            o = json.loads(body.decode('utf-8'))
        except Exception as e:
            raise FetchError(f'invalid weather JSON: {e}')
        if isinstance(o, dict) and o.get('error'):
            raise FetchError(f"Open-Meteo error: {o.get('reason')}")
        items = o if isinstance(o, list) else [o]
        if len(items) != n_locations or not all(isinstance(i, dict) and 'hourly' in i for i in items):
            raise FetchError(f'weather response has {len(items)} location blocks, expected {n_locations} with hourly data')
    return v


# ---------- date evidence ----------
def norm_id(v):
    """Exact-ID normalisation only: whitespace collapse, integral floats ('123.0', 123.0) -> '123'. No fuzzy matching."""
    if v is None:
        return ''
    if isinstance(v, (bool, np.bool_)):
        return str(v)
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        if not math.isfinite(float(v)):
            return ''
        return str(int(v)) if float(v).is_integer() else repr(float(v))
    s = re.sub(r'\s+', ' ', str(v)).strip()
    m = re.fullmatch(r'(\d+)\.0+', s)
    return m.group(1) if m else s


def csv_date_candidates(raw, year):
    """Readings of one original CSV date string: dict(kind, mdy, dmy, candidates in `year`). Never chooses between readings."""
    s = re.sub(r'\s+', ' ', '' if raw is None else str(raw)).strip()
    out = {'kind': 'missing', 'mdy': None, 'dmy': None, 'candidates': ()}
    if not s:
        return out
    if re.fullmatch(r'\d+(\.0+)?', s):
        n = int(float(s))
        if 30000 <= n <= 60000:
            d = EXCEL_ORIGIN + dt.timedelta(days=n)
            out.update(kind='excel_serial', candidates=(d,) if d.year == year else ())
        else:
            out['kind'] = 'invalid_number'
        return out
    m = re.fullmatch(r'(\d{4})-(\d{1,2})-(\d{1,2})', s)
    if m:
        try:
            d = dt.date(*map(int, m.groups()))
            out.update(kind='iso', candidates=(d,) if d.year == year else ())
        except ValueError:
            out['kind'] = 'invalid_iso'
        return out
    m = re.fullmatch(r'(\d{1,2})/(\d{1,2})/(\d{4})', s)
    if m:
        a, b, y = map(int, m.groups())
        y = y - 543 if y > 2400 else y

        def mk(mm, dd):
            try:
                return dt.date(y, mm, dd)
            except ValueError:
                return None
        mdy, dmy = mk(a, b), mk(b, a)
        cands = tuple(sorted({d for d in (mdy, dmy) if d is not None and d.year == year}))
        out.update(kind='slash', mdy=mdy, dmy=dmy, candidates=cands)
        return out
    out['kind'] = 'unrecognized'
    return out


def date_format_profile(series, year):
    """Per-file format profile of raw date strings (evidence is yearly, never one global rule)."""
    kinds, shapes = {}, {'first_gt12': 0, 'second_gt12': 0, 'equal_parts': 0, 'both_le12_ambiguous': 0}
    for v, n in series.astype(str).value_counts().items():
        c = csv_date_candidates(v, year)
        kinds[c['kind']] = kinds.get(c['kind'], 0) + int(n)
        m = re.fullmatch(r'\s*(\d{1,2})/(\d{1,2})/\d{4}\s*', v)
        if m:
            a, b = int(m.group(1)), int(m.group(2))
            key = 'equal_parts' if a == b else ('first_gt12' if a > 12 else ('second_gt12' if b > 12 else 'both_le12_ambiguous'))
            shapes[key] += int(n)
    return {'csv_format_summary': ';'.join(f'{k}:{v}' for k, v in sorted(kinds.items())),
            'csv_excel_serial_rows': kinds.get('excel_serial', 0), 'csv_slash_rows': kinds.get('slash', 0),
            'csv_ambiguous_slash_rows': shapes['both_le12_ambiguous'], 'csv_slash_first_part_gt12_rows': shapes['first_gt12'],
            'csv_slash_second_part_gt12_rows': shapes['second_gt12'], 'csv_slash_equal_parts_rows': shapes['equal_parts']}


def classify_xlsx_date(id_value, date_value, xlsx_row):
    if isinstance(date_value, dt.datetime):
        kind, d = 'typed_datetime', date_value.date().isoformat()
    elif isinstance(date_value, dt.date):
        kind, d = 'typed_date', date_value.isoformat()
    elif date_value is None or (isinstance(date_value, str) and not date_value.strip()):
        kind, d = 'missing', None
    elif isinstance(date_value, (int, float)) and not isinstance(date_value, bool):
        kind, d = 'untyped_numeric', None  # a plain number is not typed date evidence
    else:
        kind, d = 'text', None
    return {'acc_code': norm_id(id_value), 'xlsx_date': d, 'xlsx_kind': kind, 'xlsx_row': xlsx_row, 'xlsx_raw': str(date_value)[:40]}


def read_xlsx_id_dates(path, id_header='ACC_CODE', date_header='วันที่เกิดเหตุ', max_header_scan=30):
    """(DataFrame acc_code/xlsx_date/xlsx_kind/xlsx_row, meta) from the first sheet whose header row has both columns."""
    from openpyxl import load_workbook
    wb = load_workbook(path, read_only=True, data_only=True)
    try:
        for ws in wb.worksheets:
            it = ws.iter_rows(values_only=True)
            for i, row in enumerate(it):
                if i >= max_header_scan:
                    break
                names = [re.sub(r'\s+', '', str(c)) if c is not None else '' for c in (row or ())]
                if id_header in names and date_header in names:
                    ci, cd = names.index(id_header), names.index(date_header)
                    recs, blank = [], 0
                    for j, r in enumerate(it):
                        if r is None or all(c is None or (isinstance(c, str) and not c.strip()) for c in r):
                            blank += 1
                            continue
                        recs.append(classify_xlsx_date(r[ci] if ci < len(r) else None, r[cd] if cd < len(r) else None, i + j + 2))
                    df = pd.DataFrame(recs, columns=['acc_code', 'xlsx_date', 'xlsx_kind', 'xlsx_row', 'xlsx_raw'])
                    return df, {'sheet': ws.title, 'sheets': list(wb.sheetnames), 'header_row': i + 1, 'blank_rows_skipped': blank,
                                'rows': len(df), 'kind_counts': df['xlsx_kind'].value_counts().to_dict()}
        raise ValueError(f'{path}: no header row with {id_header!r} and {date_header!r} in the first {max_header_scan} rows of any sheet')
    finally:
        wb.close()


def audit_dates_by_id(csv_rows, xlsx_rows, year):
    """csv_rows: source_row, acc_code, date_raw (original v1 CSV). xlsx_rows: acc_code, xlsx_date (ISO or None), xlsx_kind.
    Exact ID join only. A CSV row is 'resolved' only when every XLSX row of its ID carries the same TYPED date, every CSV row
    of the ID has the same raw string, and that date is one of the row's own candidate readings. Returns (rows, summary)."""
    c = csv_rows[['source_row', 'acc_code', 'date_raw']].copy()
    c['acc_code'] = c['acc_code'].map(norm_id)
    parsed = {v: csv_date_candidates(v, year) for v in c['date_raw'].unique()}
    x = xlsx_rows.copy()
    x['acc_code'] = x['acc_code'].map(norm_id)
    x = x[x['acc_code'] != '']
    xg = {}
    for k, g in x.groupby('acc_code', sort=False):
        typed = g[g['xlsx_kind'].isin(TYPED_KINDS)]
        xg[k] = (len(g), set(typed['xlsx_date']), len(typed) < len(g))
    with_id = c[c['acc_code'] != '']
    csv_n = with_id.groupby('acc_code').size().to_dict()
    csv_raw_n = with_id.groupby('acc_code')['date_raw'].nunique().to_dict()
    out = {'csv_kind': [], 'ambiguous_in_v1': [], 'resolution_status': [], 'xlsx_reading': [], 'resolved_date': [],
           'xlsx_rows_for_id': [], 'csv_rows_for_id': []}
    for a, raw in zip(c['acc_code'], c['date_raw']):
        p = parsed[raw]
        st, rd, rv, nx = None, None, None, 0
        if a == '':
            st = 'no_id'
        elif a not in xg:
            st = 'id_absent_in_xlsx'
        else:
            nx, dates, untyped = xg[a]
            if untyped:
                st = 'xlsx_untyped_or_missing_date'
            elif len(dates) != 1:
                st = 'xlsx_id_conflicting_dates'
            elif csv_raw_n[a] > 1:
                st = 'csv_id_conflicting_dates'
            elif not p['candidates']:
                st = 'csv_date_unparseable'
            else:
                xd = dt.date.fromisoformat(next(iter(dates)))
                if xd not in p['candidates']:
                    st = 'mismatch'
                else:
                    st, rv = 'resolved', xd.isoformat()
                    if p['kind'] != 'slash':
                        rd = 'non_slash'
                    elif p['mdy'] == p['dmy']:
                        rd = 'equal_parts'
                    else:
                        rd = 'mdy' if xd == p['mdy'] else 'dmy'
        out['csv_kind'].append(p['kind'])
        out['ambiguous_in_v1'].append(len(p['candidates']) > 1)
        out['resolution_status'].append(st)
        out['xlsx_reading'].append(rd)
        out['resolved_date'].append(rv)
        out['xlsx_rows_for_id'].append(nx)
        out['csv_rows_for_id'].append(csv_n.get(a, 0) if a else 0)
    for k, v in out.items():
        c[k] = v
    sc = c['resolution_status'].value_counts().to_dict()
    rc = c.loc[c['resolution_status'] == 'resolved', 'xlsx_reading'].value_counts().to_dict()
    n_with_id = int((c['acc_code'] != '').sum())
    n_mdy, n_dmy, n_mis = rc.get('mdy', 0), rc.get('dmy', 0), sc.get('mismatch', 0)
    if n_mdy and not n_dmy and not n_mis and sc.get('resolved', 0) == n_with_id:
        concl = 'mdy_all_id_rows_agree'
    elif n_mdy and not n_dmy:
        concl = 'mdy_only_among_resolved_some_rows_unresolved'
    elif n_dmy and not n_mdy:
        concl = 'dmy_only_among_resolved' + ('' if sc.get('resolved', 0) == n_with_id else '_some_rows_unresolved')
    elif n_mdy and n_dmy:
        concl = 'mixed_readings_per_id_resolution_only'
    else:
        concl = 'no_distinguishing_evidence'
    csv_ids, x_ids = set(csv_n), set(xg)
    amb = c['ambiguous_in_v1']
    summary = {
        'n_csv_rows': len(c), 'n_csv_rows_with_id': n_with_id, 'n_csv_unique_ids': len(csv_ids),
        'n_csv_duplicate_id_groups': sum(1 for v in csv_n.values() if v > 1),
        'n_csv_ids_conflicting_dates': sum(1 for v in csv_raw_n.values() if v > 1),
        'n_xlsx_rows': len(xlsx_rows), 'n_xlsx_rows_with_id': len(x), 'n_xlsx_unique_ids': len(x_ids),
        'n_xlsx_duplicate_id_groups': sum(1 for v in xg.values() if v[0] > 1),
        'xlsx_kind_counts': xlsx_rows['xlsx_kind'].value_counts().to_dict(),
        'n_ids_csv_only': len(csv_ids - x_ids), 'n_ids_xlsx_only_not_added': len(x_ids - csv_ids),
        'n_ids_row_count_differs': sum(1 for k in csv_ids & x_ids if csv_n[k] != xg[k][0]),
        'status_counts': sc, 'reading_counts': rc, 'n_resolved': sc.get('resolved', 0), 'n_mismatch': n_mis,
        'n_ambiguous_in_v1': int(amb.sum()), 'n_ambiguous_resolved': int((amb & (c['resolution_status'] == 'resolved')).sum()),
        'n_ambiguous_unresolved': int((amb & (c['resolution_status'] != 'resolved')).sum()), 'conclusion': concl,
    }
    return c, summary


# ---------- traffic (DOH AADT) ----------
def resource_meta(res):
    res = res or {}
    return {'resource_id': res.get('id'), 'resource_name': res.get('name'), 'format': res.get('format'), 'url': res.get('url'),
            'catalog_created': res.get('created'), 'catalog_last_modified': res.get('last_modified'),
            'catalog_metadata_modified': res.get('metadata_modified'),
            'catalog_data_release': res.get('data_release_date') or res.get('resource_data_release_calendar'),
            'catalog_resource_last_updated': res.get('resource_last_updated_date'), 'catalog_size': res.get('size'),
            'catalog_hash': res.get('hash') or None, 'catalog_unit': res.get('resource_unit_of_measure'),
            'available_from_proxy': res.get('created'),
            'available_from_note': 'catalog created timestamp is a publication proxy, not proof the current bytes existed then'}


def resource_year_be(res):
    m = re.search(r'(25\d\d)', str(res.get('name') or ''))
    if m:
        return int(m.group(1))
    m = re.search(r'aadt[-_](\d\d)[-_.]', str(res.get('url') or '').lower())
    return 2500 + int(m.group(1)) if m else None


def select_aadt_resources(resources, years_be, prefer_format='CSV'):
    """{be_year: (chosen resource, [alternatives])}; latest 'created' wins among same-year resources of the preferred format."""
    out = {}
    for be in years_be:
        cands = [r for r in resources if str(r.get('format', '')).upper() == prefer_format.upper() and resource_year_be(r) == be]
        if cands:
            cands = sorted(cands, key=lambda r: str(r.get('created') or ''), reverse=True)
            out[be] = (cands[0], cands[1:])
    return out


def read_csv_bytes(body, encodings):
    for enc in encodings:
        try:
            return pd.read_csv(io.BytesIO(body), dtype=str, keep_default_na=False, encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f'CSV not decodable with {encodings}')


def norm_header(h):
    return re.sub(r'\s+', '', str('' if h is None else h)).replace('﻿', '')


def to_number(s):
    """'1,234' -> 1234.0; '', '-', text -> NaN (never 0)."""
    t = s.astype(str).str.replace(',', '', regex=False).str.strip()
    return pd.to_numeric(t.where(~t.isin(['', '-', 'nan', 'None', 'NaN'])), errors='coerce')


def normalize_aadt(raw, aliases, required, total_components, heavy_components, be_year):
    hdr = {}
    for c in raw.columns:
        hdr.setdefault(norm_header(c), c)
    mapped = {}
    for canon, names in aliases.items():
        hit = [hdr[norm_header(n)] for n in names if norm_header(n) in hdr]
        if hit:
            mapped[canon] = hit[0]
    missing = [k for k in required if k not in mapped]
    out = pd.DataFrame({'survey_year_be': be_year, 'survey_year_ce': be_year - 543}, index=raw.index)
    for canon in ('province', 'route', 'control_section', 'survey_point', 'route_name', 'highway_district'):
        out[canon] = raw[mapped[canon]].astype(str).str.strip() if canon in mapped else ''
    out['total_aadt'] = to_number(raw[mapped['total']]) if 'total' in mapped else np.nan
    out['heavy_share_pct'] = to_number(raw[mapped['heavy_share_pct']]) if 'heavy_share_pct' in mapped else np.nan
    used = set(mapped.values())
    nonnumeric, unmapped_text = {}, []
    for c in raw.columns:
        if c in used:
            continue
        v = to_number(raw[c])
        nonempty = raw[c].astype(str).str.strip().ne('')
        if nonempty.any() and v.notna().sum() >= 0.5 * nonempty.sum():
            out['count__' + norm_header(c)] = v
            nonnumeric[norm_header(c)] = int((nonempty & v.isna()).sum())
        else:
            unmapped_text.append(norm_header(c))
    for c in ('total_aadt', 'heavy_share_pct'):
        src = mapped.get('total' if c == 'total_aadt' else 'heavy_share_pct')
        if src is not None:
            ne = raw[src].astype(str).str.strip().ne('')
            nonnumeric[c] = int((ne & out[c].isna()).sum())
    comp = ['count__' + norm_header(n) for n in total_components]
    if all(c in out for c in comp) and 'total' in mapped:
        diff = (out[comp].sum(axis=1, min_count=len(comp)) - out['total_aadt']).abs()
        total_check = {'evaluated': True, 'n_compared': int(diff.notna().sum()), 'n_mismatch_gt1': int((diff > 1).sum())}
    else:
        total_check = {'evaluated': False, 'missing_components': [c for c in comp if c not in out], 'n_mismatch_gt1': None}
    hv = ['count__' + norm_header(n) for n in heavy_components]
    if all(c in out for c in hv) and 'total' in mapped and 'heavy_share_pct' in mapped:
        share = 100.0 * out[hv].sum(axis=1, min_count=len(hv)) / out['total_aadt'].where(out['total_aadt'] > 0)
        d = (share - out['heavy_share_pct']).abs()
        heavy_check = {'evaluated': True, 'n_compared': int(d.notna().sum()), 'n_mismatch_gt0_05pp': int((d > 0.05).sum())}
    else:
        heavy_check = {'evaluated': False, 'n_mismatch_gt0_05pp': None}
    key = ['route', 'control_section', 'survey_point']
    dup = int(out.duplicated(key, keep=False).sum()) if all(k in mapped for k in key) else None
    num_cols = [c for c in out.columns if c.startswith('count__')] + ['total_aadt']
    neg = int((out[num_cols] < 0).sum().sum())
    rep = {'status': 'normalized' if not missing else 'schema_incomplete', 'mapped': mapped, 'missing_fields': missing,
           'unmapped_text_headers': unmapped_text, 'duplicate_key_rows': dup, 'total_check': total_check, 'heavy_check': heavy_check,
           'negative_values': neg, 'nonnumeric_cells': nonnumeric, 'unit': 'vehicles/day (AADT) at a survey point'}
    return out, rep


# ---------- geography ----------
def norm_en(s):
    return re.sub(r'[^a-z0-9]', '', str(s or '').lower())


def _pick_key(keys, candidates):
    low = {str(k).lower(): k for k in keys}
    for c in candidates:
        if c.lower() in low:
            return low[c.lower()]
    return None


def province_table(obj):
    rows = obj
    if isinstance(obj, dict):
        rows = next((obj[k] for k in ('data', 'provinces', 'items') if isinstance(obj.get(k), list)), None)
    if not isinstance(rows, list) or not rows or not isinstance(rows[0], dict):
        raise ValueError('province table: expected a JSON list of objects')
    keys = list(rows[0].keys())
    kc = _pick_key(keys, ['provinceCode', 'province_code', 'code', 'PROVINCE_CODE'])
    kt = _pick_key(keys, ['provinceNameTh', 'province_name_th', 'nameTh', 'name_th'])
    ke = _pick_key(keys, ['provinceNameEn', 'province_name_en', 'nameEn', 'name_en'])
    if not (kc and kt and ke):
        raise ValueError(f'province table keys not recognised (need code/Thai/English name): {sorted(keys)}')
    df = pd.DataFrame({'province_code': [int(r[kc]) for r in rows], 'name_th': [str(r[kt]).strip() for r in rows],
                       'name_en': [str(r[ke]).strip() for r in rows]})
    if df['province_code'].duplicated().any():
        raise ValueError('province table has duplicate province codes')
    return df, {'code_key': kc, 'name_th_key': kt, 'name_en_key': ke}


def join_shapes(shapes, provinces):
    """ISO 3166-2 'TH-NN' -> province code first, else EXACT normalized English name; never fuzzy. Unresolved stay explicit."""
    codes = set(provinces['province_code'])
    by_name = {}
    for code, en in zip(provinces['province_code'], provinces['name_en']):
        by_name.setdefault(norm_en(en), []).append(code)
    recs = []
    for idx, name, iso in zip(shapes['shape_index'], shapes['shape_name'], shapes['shape_iso']):
        m = re.fullmatch(r'TH-(\d{2})', str(iso or '').strip())
        code, how = None, 'unresolved_no_iso_or_exact_name'
        if m and int(m.group(1)) in codes:
            code, how = int(m.group(1)), 'iso_3166_2_code'
        else:
            hits = by_name.get(norm_en(name), [])
            if len(hits) == 1:
                code, how = hits[0], 'exact_normalized_english_name'
            elif len(hits) > 1:
                how = 'unresolved_name_not_unique'
        recs.append({'shape_index': idx, 'province_code': code, 'join_method': how})
    j = pd.DataFrame(recs, columns=['shape_index', 'province_code', 'join_method'])
    dup = j['province_code'].notna() & j['province_code'].duplicated(keep=False)
    j.loc[dup, 'join_method'] = 'unresolved_duplicate_code'
    j.loc[dup, 'province_code'] = None
    j['province_code'] = pd.array(j['province_code'], dtype='Int64')
    return j


def representative_points(features):
    from shapely.geometry import shape
    recs = []
    for i, f in enumerate(features):
        props = f.get('properties') or {}
        rec = {'shape_index': i, 'shape_name': props.get('shapeName'), 'shape_iso': props.get('shapeISO'), 'shape_id': props.get('shapeID'),
               'lat': None, 'lon': None, 'point_within_polygon': False, 'geometry_valid': False}
        if f.get('geometry'):
            g = shape(f['geometry'])
            rec['geometry_valid'] = bool(g.is_valid)
            if not g.is_empty:
                p = g.representative_point()
                rec.update(lat=float(p.y), lon=float(p.x), point_within_polygon=bool(g.contains(p)))
        recs.append(rec)
    return pd.DataFrame(recs)


# ---------- weather (archived forecasts) ----------
def estimate_call_weight(n_locations, n_days, n_vars):
    """Approximate Open-Meteo fair-use weight (locations x 2-week units x variable factor); an estimate, not the official count."""
    return round(n_locations * max(1.0, n_days / 14.0) * max(1.0, n_vars / 10.0), 1)


def weather_url(base, batch, start, end, w):
    q = {'latitude': ','.join(f'{v:.4f}' for v in batch['lat']), 'longitude': ','.join(f'{v:.4f}' for v in batch['lon']),
         'hourly': ','.join(w['hourly_variables']), 'models': w['model'], 'timezone': w['timezone'], 'start_date': start, 'end_date': end}
    return base + '?' + urlencode(q, safe=',')


def normalize_weather_payload(payload, batch, variables, model, timezone, utc_offset_seconds, lead_hours):
    """Hourly rows per location with valid time (local + UTC) and the conservative nominal issue bound = valid UTC - lead."""
    items = payload if isinstance(payload, list) else [payload]
    if len(items) != len(batch):
        raise ValueError(f'{len(items)} location blocks for {len(batch)} requested points')
    frames = []
    for it, pt in zip(items, batch.itertuples(index=False)):
        if it.get('error'):
            raise ValueError(f"Open-Meteo error: {it.get('reason')}")
        if it.get('utc_offset_seconds') != utc_offset_seconds or it.get('timezone') != timezone:
            raise ValueError(f"unexpected timezone {it.get('timezone')}/{it.get('utc_offset_seconds')} (expected {timezone}/{utc_offset_seconds})")
        h = it['hourly']
        t = pd.to_datetime(pd.Series(h['time']), format='%Y-%m-%dT%H:%M')
        f = pd.DataFrame({'province_code': int(pt.province_code), 'valid_time_local': t})
        for v in variables:
            key = v if v in h else next((k for k in h if k.startswith(v)), None)
            if key is None:
                raise ValueError(f'variable {v} missing from response')
            vals = pd.to_numeric(pd.Series(h[key], dtype=object), errors='coerce')
            if len(vals) != len(t):
                raise ValueError(f'{v}: {len(vals)} values for {len(t)} timestamps')
            f[v] = vals.to_numpy(dtype=float)
        f['valid_time_utc'] = f['valid_time_local'] - pd.Timedelta(seconds=utc_offset_seconds)
        f['nominal_issue_bound_utc'] = f['valid_time_utc'] - pd.Timedelta(hours=lead_hours)
        f['requested_lat'], f['requested_lon'] = float(pt.lat), float(pt.lon)
        f['grid_lat'], f['grid_lon'], f['grid_elevation'] = it.get('latitude'), it.get('longitude'), it.get('elevation')
        f['timezone'], f['utc_offset_seconds'], f['model'], f['lead_hours'] = it.get('timezone'), it.get('utc_offset_seconds'), model, lead_hours
        frames.append(f)
    return pd.concat(frames, ignore_index=True)


def aggregate_weather(hourly, block_hours, temp_col, rain_col, buffer_hours, issue_hour_local, utc_offset_seconds):
    """Per province x local date x block (24 = daily, 6 = 00-06/06-12/12-18/18-24). Sums/means only from COMPLETE hours
    (all-null is never 0). issue_bound_ok: latest nominal issue bound + buffer <= (target date - 1) at issue_hour_local."""
    h = hourly.copy()
    h['date'] = h['valid_time_local'].dt.normalize()
    h['block'] = h['valid_time_local'].dt.hour // block_hours
    g = h.groupby(['province_code', 'date', 'block'])
    a = g.agg(n_rows=('valid_time_local', 'size'), n_hours=('valid_time_local', 'nunique'), n_valid_rain=(rain_col, 'count'),
              n_valid_temp=(temp_col, 'count'), _rain=(rain_col, 'sum'), _temp=(temp_col, 'mean'),
              earliest_issue_bound_utc=('nominal_issue_bound_utc', 'min'), latest_issue_bound_utc=('nominal_issue_bound_utc', 'max')).reset_index()
    a['n_expected'] = block_hours
    clean = (a['n_rows'] == block_hours) & (a['n_hours'] == block_hours)
    a['rain_missing'] = ~(clean & (a['n_valid_rain'] == block_hours))
    a['temp_missing'] = ~(clean & (a['n_valid_temp'] == block_hours))
    a['rain_sum'] = a['_rain'].where(~a['rain_missing'])
    a['temp_mean'] = a['_temp'].where(~a['temp_missing'])
    a = a.drop(columns=['_rain', '_temp'])
    a['time_block'] = 'day' if block_hours == 24 else a['block'].map(lambda b: f'{b * block_hours:02d}-{(b + 1) * block_hours:02d}')
    issue_local = a['date'] - pd.Timedelta(days=1) + pd.Timedelta(hours=issue_hour_local)
    a['issue_time_utc'] = issue_local - pd.Timedelta(seconds=utc_offset_seconds)
    a['issue_bound_ok'] = (a['latest_issue_bound_utc'] + pd.Timedelta(hours=buffer_hours)) <= a['issue_time_utc']
    return a


# ---------- calendar ----------
def html_to_text(html):
    s = re.sub(r'(?is)<(script|style)\b.*?</\1>', ' ', html)
    s = re.sub(r'(?i)<(br|/p|/div|/li|/tr|/h\d)[^>]*>', '\n', s)
    s = re.sub(r'<[^>]+>', ' ', s)
    return '\n'.join(re.sub(r'[ \t\r\f\v]+', ' ', ln).strip() for ln in htmllib.unescape(s).splitlines() if ln.strip())


def discover_pdf_links(html, base_url, allowed_hosts):
    out = []
    for h in re.findall(r'''href\s*=\s*["']([^"']+?\.pdf)(?:[?#][^"']*)?["']''', html, flags=re.I):
        u = urljoin(base_url, htmllib.unescape(h))
        if 'holiday' in u.lower() and host_allowed(u, allowed_hosts) and u not in out:
            out.append(u)
    return out


def year_from_url(u):
    name = urlparse(u).path.rsplit('/', 1)[-1]
    m = re.search(r'(20\d\d)', name)
    if m:
        return int(m.group(1))
    m = re.search(r'(25\d\d)', name)
    return int(m.group(1)) - 543 if m else None


def pdf_text(body):
    from pypdf import PdfReader
    r = PdfReader(io.BytesIO(body))
    return '\n'.join((p.extract_text() or '') for p in r.pages), len(r.pages)


def extract_thai_dates(text, year):
    """[(date, line)] for every '<day> <Thai month>' mention (Thai or Arabic digits) in a source for calendar `year`."""
    pat = re.compile(r'(\d{1,2})\s*(' + '|'.join(THAI_MONTHS) + r')')
    out = []
    for line in str(text).translate(THAI_DIGITS).splitlines():
        for m in pat.finditer(line):
            try:
                d = dt.date(int(year), THAI_MONTHS[m.group(2)], int(m.group(1)))
            except ValueError:
                continue
            out.append({'date': d.isoformat(), 'line': re.sub(r'\s+', ' ', line).strip()[:200]})
    return out


def classify_calendar_records(found, recurring, src):
    """Only recurring fixed festivals whose keyword is on the same line are verified; everything else is explicitly excluded.
    Bank (BOT) holidays are never labelled national public holidays."""
    recs = []
    for f in found:
        d = dt.date.fromisoformat(f['date'])
        fest = next((r for r in recurring if [d.month, d.day] in [list(md) for md in r['month_days']]
                     and any(k in f['line'] for k in r['keywords'])), None)
        recs.append({'date': f['date'], 'name': fest['name'] if fest else None, 'festival_id': fest['id'] if fest else None,
                     'status': VERIFIED_FESTIVAL if fest else EXCLUDED_TEXT, 'record_type': src.get('record_type', 'bot_financial_institution_holiday'),
                     'is_national_public_holiday': None, 'source_id': src.get('source_id'), 'source_url': src.get('url'),
                     'source_sha256': src.get('sha256'), 'source_year': src.get('year'), 'announcement_date': src.get('announcement_date'),
                     'announcement_date_status': src.get('announcement_date_status', 'unknown'), 'source_text': f['line']})
    return recs


def dedupe_calendar_records(df):
    return df.drop_duplicates(['date', 'festival_id', 'status', 'source_id', 'source_text']).sort_values(['date', 'source_id']).reset_index(drop=True)


def festival_evidence(records, recurring, years):
    rows = []
    for y in years:
        for r in recurring:
            ok = records[(records['status'] == VERIFIED_FESTIVAL) & (records['festival_id'] == r['id'])
                         & (records['record_type'] == 'bot_financial_institution_holiday') & (records['source_year'] == y)]
            rows.append({'year': y, 'festival_id': r['id'], 'name': r['name'],
                         'rule_dates': ';'.join(dt.date(y, m, d).isoformat() for m, d in r['month_days']),
                         'confirmed_in_official_pdf': len(ok) > 0, 'confirmed_dates': ';'.join(sorted(set(ok['date']))),
                         'sources': ';'.join(sorted(set(ok['source_id'].astype(str))))})
    return pd.DataFrame(rows, columns=['year', 'festival_id', 'name', 'rule_dates', 'confirmed_in_official_pdf', 'confirmed_dates', 'sources'])


def calendar_feature_eligible(rec):
    """Next-stage guard: only verified recurring fixed festivals may become features (never generic bank holidays)."""
    return rec.get('status') == VERIFIED_FESTIVAL and bool(rec.get('festival_id'))


def announcement_known_before(rec, issue_date):
    """Announced/special days are usable only when a verified announcement date is on or before the issue date."""
    a = rec.get('announcement_date')
    return bool(a) and dt.date.fromisoformat(str(a)) <= issue_date

In [2]:
# FUNCTIONS (part 2: reviewed calendar, date-rule agreement/context, traffic rows/availability, pinned geography, exact assets) - tests execute both FUNCTIONS cells
CAL_ACTIONS = ('add', 'remove')
CAL_RECORD_COLUMNS = ['date', 'name', 'action', 'scope', 'is_national_public_holiday', 'source_id', 'source_kind', 'source_role',
                      'source_url', 'parent_url', 'page_ref', 'known_from', 'available_from_local', 'verified_by',
                      'source_review_status', 'applied', 'exclusion_reason', 'reviewed_copy_sha256', 'live_sha256']


def functions_cells_source(nb_path):
    """Concatenated source of every '# FUNCTIONS' code cell in notebook order (hashed into the manifest)."""
    nb = json.loads(Path(nb_path).read_text('utf-8'))
    srcs = [''.join(c['source']) if isinstance(c['source'], list) else c['source'] for c in nb['cells'] if c['cell_type'] == 'code']
    hits = [s for s in srcs if s.lstrip().startswith('# FUNCTIONS')]
    if not hits:
        raise ValueError(f'{nb_path}: no FUNCTIONS cell')
    return '\n\n'.join(hits)


# ---------- exact official embedded assets ----------
def exact_asset_net(net, url, assets):
    """net settings that additionally allow ONE configured asset URL (exact match) whose parent page is on an allowlisted host."""
    hit = next((a for a in assets if a.get('url') == url), None)
    if hit is None or not hit.get('parent_url') or not host_allowed(hit['parent_url'], net['allowed_hosts']):
        raise FetchError(f'not a configured exact official asset with an allowlisted parent page: {url}')
    return dict(net, allowed_hosts=list(net['allowed_hosts']) + [urlparse(url).hostname])


def png_validator(body):
    if not body.startswith(b'\x89PNG\r\n\x1a\n'):
        raise FetchError('response is not a PNG image')


# ---------- weather payload validation (before normalisation) ----------
WEATHER_NEGATIVE_POLICIES = ('strict', 'mask')


def validate_weather_payload(payload, batch, variables, units, start, end, nonneg_vars, max_grid_deg, negative_policy='strict'):
    """Strict checks: one block per requested point, EXACT hourly keys (no prefix matches) with expected units, a unique regular
    hourly grid covering exactly start..end local, numeric finite values (null allowed = coverage gap), plausible grid coordinates
    within max_grid_deg of the request. Raises ValueError. A negative finite value of a nonneg_var raises under 'strict'; under 'mask'
    it is a PER-HOUR value-quality error left to apply_weather_value_policy (every structural check above stays strict)."""
    if negative_policy not in WEATHER_NEGATIVE_POLICIES:
        raise ValueError(f'negative_policy {negative_policy!r} not in {WEATHER_NEGATIVE_POLICIES}')
    items = payload if isinstance(payload, list) else [payload]
    if len(items) != len(batch):
        raise ValueError(f'{len(items)} location blocks for {len(batch)} requested points')
    expected = pd.date_range(f'{start} 00:00', f'{end} 23:00', freq='h')
    for it, pt in zip(items, batch.itertuples(index=False)):
        h, u = it.get('hourly') or {}, it.get('hourly_units') or {}
        missing, extra = [v for v in variables if v not in h], sorted(set(h) - set(variables) - {'time'})
        if missing or extra:
            raise ValueError(f'hourly keys: missing {missing}, unexpected {extra}')
        bad_units = {v: u.get(v) for v, unit in units.items() if u.get(v) != unit}
        if bad_units:
            raise ValueError(f'unexpected hourly units {bad_units} (expected {units})')
        t = pd.to_datetime(pd.Series(h.get('time', [])), format='%Y-%m-%dT%H:%M')
        if len(t) != len(expected) or t.duplicated().any() or not (t.to_numpy() == expected.to_numpy()).all():
            raise ValueError(f'hourly time grid is not exactly {start}..{end} hourly ({len(t)} stamps)')
        for v in variables:
            s = pd.Series(h[v], dtype=object)
            if len(s) != len(t):
                raise ValueError(f'{v}: {len(s)} values for {len(t)} timestamps')
            vals = pd.to_numeric(s.dropna(), errors='coerce')
            if vals.isna().any() or np.isinf(vals.astype(float)).any():
                raise ValueError(f'{v}: non-numeric or non-finite values')
            if v in nonneg_vars and negative_policy == 'strict' and (vals < 0).any():
                raise ValueError(f'{v}: negative values')
        lat, lon = it.get('latitude'), it.get('longitude')
        if not (isinstance(lat, (int, float)) and isinstance(lon, (int, float)) and -90 <= lat <= 90 and -180 <= lon <= 180
                and abs(lat - float(pt.lat)) <= max_grid_deg and abs(lon - float(pt.lon)) <= max_grid_deg):
            raise ValueError(f'grid coordinates {lat},{lon} implausible for requested {pt.lat:.4f},{pt.lon:.4f}')


WEATHER_QUALITY_FLAGS = ('valid', 'null_in_source', 'negative_value_masked')
WEATHER_QUALITY_COLUMNS = ['province_code', 'variable', 'n_returned_hours', 'n_valid', 'n_null_in_source', 'n_negative_masked',
                           'min_raw', 'max_raw', 'quality_reason']


def apply_weather_value_policy(hourly, variables, nonneg_vars):
    """Per-hour value-quality policy on a normalized (validated) frame, in memory only (raw JSON bytes are never touched).
    A negative finite value of a nonneg_var becomes NaN in that hour ONLY (never abs/clip/0/any correction); every other hour, every
    other variable (temperature) and every location are kept. Adds '<var>_quality' in WEATHER_QUALITY_FLAGS per variable.
    Returns (masked frame, hour-level flags of masked hours with the raw value, per province x variable audit with raw min/max)."""
    h = hourly.copy()
    flags, audit = [], []
    for v in variables:
        raw = h[v].astype(float)
        neg = raw.notna() & (raw < 0) if v in nonneg_vars else pd.Series(False, index=h.index)
        q = pd.Series('valid', index=h.index, dtype=object)
        q[raw.isna()] = 'null_in_source'
        q[neg] = 'negative_value_masked'
        h[v], h[v + '_quality'] = raw.mask(neg), q
        flags.append(pd.DataFrame({'province_code': h.loc[neg, 'province_code'], 'valid_time_local': h.loc[neg, 'valid_time_local'],
                                   'variable': v, 'raw_value': raw[neg], 'quality_flag': 'negative_value_masked'}))
        g = pd.DataFrame({'province_code': h['province_code'], 'raw': raw, 'null': raw.isna(), 'neg': neg}).groupby('province_code')
        a = g.agg(n_returned_hours=('raw', 'size'), n_null_in_source=('null', 'sum'), n_negative_masked=('neg', 'sum'),
                  min_raw=('raw', 'min'), max_raw=('raw', 'max')).reset_index()
        a['n_valid'] = a['n_returned_hours'] - a['n_null_in_source'] - a['n_negative_masked']
        a['variable'] = v
        a['quality_reason'] = [';'.join(r for r, n in (('negative_value_masked_to_nan', m), ('null_in_source', z)) if n) or 'ok'
                               for m, z in zip(a['n_negative_masked'], a['n_null_in_source'])]
        audit.append(a)
    fl = pd.concat(flags, ignore_index=True).sort_values(['province_code', 'valid_time_local', 'variable'], ignore_index=True)
    au = pd.concat(audit, ignore_index=True)[WEATHER_QUALITY_COLUMNS]
    for c in ('n_returned_hours', 'n_valid', 'n_null_in_source', 'n_negative_masked'):
        au[c] = au[c].astype(int)
    return h, fl, au


# ---------- date evidence: supporting context and both global readings (report only) ----------
def norm_text(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return ''
    return re.sub(r'\s+', ' ', str(v)).strip()


def norm_time(v):
    """'HH:MM' from a typed time/datetime, an Excel day fraction or text like '9:05', '09.05 น.', '09:05:00'; otherwise ''."""
    if v is None:
        return ''
    if isinstance(v, dt.datetime):
        v = v.time()
    if isinstance(v, dt.time):
        return f'{v.hour:02d}:{v.minute:02d}'
    if isinstance(v, (float, np.floating)) and 0 <= float(v) < 1:
        m = int(round(float(v) * 1440)) % 1440
        return f'{m // 60:02d}:{m % 60:02d}'
    m = re.search(r'(\d{1,2})[:.](\d{2})', str(v))
    if m and int(m.group(1)) < 24 and int(m.group(2)) < 60:
        return f'{int(m.group(1)):02d}:{int(m.group(2)):02d}'
    return ''


def norm_context(v, kind):
    return norm_time(v) if kind == 'time' else norm_text(v)


def read_xlsx_context(path, id_header, date_header, fields, max_header_scan=30):
    """(DataFrame acc_code + normalized context fields, meta) using the same sheet/header detection as read_xlsx_id_dates.
    A field whose header is absent is None (unavailable), never guessed."""
    from openpyxl import load_workbook
    names_out = [f['field'] for f in fields]
    wb = load_workbook(path, read_only=True, data_only=True)
    try:
        for ws in wb.worksheets:
            it = ws.iter_rows(values_only=True)
            for i, row in enumerate(it):
                if i >= max_header_scan:
                    break
                names = [re.sub(r'\s+', '', str(c)) if c is not None else '' for c in (row or ())]
                if id_header in names and date_header in names:
                    ci = names.index(id_header)
                    cols = {f['field']: names.index(f['header']) for f in fields if f['header'] in names}
                    recs = []
                    for r in it:
                        if r is None or all(c is None or (isinstance(c, str) and not c.strip()) for c in r):
                            continue
                        rec = {'acc_code': norm_id(r[ci] if ci < len(r) else None)}
                        for f in fields:
                            j = cols.get(f['field'])
                            rec[f['field']] = None if j is None else norm_context(r[j] if j < len(r) else None, f['kind'])
                        recs.append(rec)
                    return pd.DataFrame(recs, columns=['acc_code'] + names_out), {
                        'sheet': ws.title, 'header_row': i + 1, 'missing_headers': [f['header'] for f in fields if f['field'] not in cols]}
        raise ValueError(f'{path}: no header row with {id_header!r} and {date_header!r}')
    finally:
        wb.close()


def apply_context_check(rows, csv_ctx, xlsx_ctx, fields):
    """Keep 'resolved' only if EVERY XLSX row of the ID equals the CSV row on every context field (blank == blank allowed, at least
    one non-blank value required). Missing headers -> context_unverified; any difference -> context_conflict. The ID/date outcome
    stays in id_date_status. Returns (rows, context_status counts)."""
    names = [f['field'] for f in fields]
    r = rows.merge(csv_ctx[['source_row'] + names], on='source_row', how='left', validate='one_to_one')
    x = xlsx_ctx[xlsx_ctx['acc_code'] != '']
    xg = {k: g for k, g in x.groupby('acc_code', sort=False)}
    status = []
    for a, st, *vals in zip(r['acc_code'], r['resolution_status'], *[r[n] for n in names]):
        if st != 'resolved':
            status.append('not_evaluated')
            continue
        g = xg.get(a)
        if g is None or any(g[n].isna().any() for n in names) or any(v is None or (isinstance(v, float) and math.isnan(v)) for v in vals):
            status.append('unavailable')
            continue
        csv_vals = [str(v) for v in vals]
        if not all(set(g[n].astype(str)) == {cv} for n, cv in zip(names, csv_vals)):
            status.append('conflict')
        elif not any(csv_vals):
            status.append('unavailable')
        else:
            status.append('agree')
    r['context_status'] = status
    r['id_date_status'] = r['resolution_status']
    r['typed_candidate_match'] = r['id_date_status'].eq('resolved')
    r.loc[r['context_status'] == 'conflict', 'resolution_status'] = 'context_conflict'
    r.loc[r['context_status'] == 'unavailable', 'resolution_status'] = 'context_unverified'
    r['resolved_date'] = r['resolved_date'].where(r['resolution_status'] == 'resolved', None)
    return r.drop(columns=names), r['context_status'].value_counts().to_dict()


def candidate_rule_agreement(csv_rows, xlsx_rows, year):
    """REPORT ONLY: compare both global readings (M/D, D/M) with the typed XLSX date on slash rows whose ID has exactly one typed
    XLSX date and one CSV raw string. Shows whether any single global parser would agree; never used to resolve rows."""
    c = pd.DataFrame({'acc_code': csv_rows['acc_code'].map(norm_id), 'date_raw': csv_rows['date_raw']})
    c = c[c['acc_code'] != '']
    x = xlsx_rows.assign(acc_code=xlsx_rows['acc_code'].map(norm_id))
    x = x[x['acc_code'] != '']
    xd = {}
    for k, g in x.groupby('acc_code', sort=False):
        if g['xlsx_kind'].isin(TYPED_KINDS).all() and g['xlsx_date'].nunique() == 1:
            xd[k] = dt.date.fromisoformat(g['xlsx_date'].iloc[0])
    raw_n = c.groupby('acc_code')['date_raw'].nunique().to_dict()
    keys = ['compared', 'mdy_agree', 'dmy_agree', 'neither', 'typed_in_candidates', 'ambiguous_compared', 'ambiguous_mdy_agree',
            'ambiguous_dmy_agree', 'ambiguous_neither', 'unambiguous_compared', 'unambiguous_agree', 'unambiguous_disagree']
    o = dict.fromkeys(keys, 0)
    for a, raw in zip(c['acc_code'], c['date_raw']):
        if a not in xd or raw_n.get(a) != 1:
            continue
        p = csv_date_candidates(raw, year)
        if p['kind'] != 'slash':
            continue
        d = xd[a]
        m, dm = p['mdy'] == d, p['dmy'] == d
        o['compared'] += 1
        o['mdy_agree'] += m
        o['dmy_agree'] += dm
        o['neither'] += not (m or dm)
        o['typed_in_candidates'] += d in p['candidates']
        if len(p['candidates']) > 1:
            o['ambiguous_compared'] += 1
            o['ambiguous_mdy_agree'] += m
            o['ambiguous_dmy_agree'] += dm
            o['ambiguous_neither'] += not (m or dm)
        else:
            o['unambiguous_compared'] += 1
            o['unambiguous_agree'] += d in p['candidates']
            o['unambiguous_disagree'] += d not in p['candidates']
    o['mdy_disagree'] = o['compared'] - o['mdy_agree']
    o['dmy_disagree'] = o['compared'] - o['dmy_agree']
    return {'rule_' + k: int(v) for k, v in o.items()}


def find_dictionary_resources(pkg, keywords):
    """Catalog resources whose name/description/url mention a data-dictionary keyword (snapshot only, not format proof)."""
    out = []
    for r in ((pkg or {}).get('result') or {}).get('resources', []):
        hay = ' '.join(str(r.get(k) or '') for k in ('name', 'description', 'url')).lower()
        if r.get('url') and any(k.lower() in hay for k in keywords):
            out.append(r)
    return out


# ---------- traffic availability and province summaries ----------
def parse_catalog_time(v):
    """CKAN timestamp (ISO, usually naive = UTC) -> aware UTC datetime, or None."""
    if not v:
        return None
    try:
        t = dt.datetime.fromisoformat(str(v).strip().replace('Z', '+00:00'))
    except ValueError:
        return None
    return t.replace(tzinfo=dt.timezone.utc) if t.tzinfo is None else t.astimezone(dt.timezone.utc)


def traffic_available_from(res, buffer_days=1):
    """Conservative catalog-vintage proxy max(created, last_modified) + buffer days; a proxy, not proof of historical delivery."""
    ts = {k: parse_catalog_time((res or {}).get(k)) for k in ('created', 'last_modified')}
    known = [t for t in ts.values() if t is not None]
    if not known:
        return {'available_from_proxy_utc': None, 'available_from_basis': 'no_catalog_created_or_last_modified'}
    basis = 'max(created,last_modified)' if len(known) == 2 else ('created_only' if ts['created'] else 'last_modified_only')
    return {'available_from_proxy_utc': (max(known) + dt.timedelta(days=int(buffer_days))).isoformat(),
            'available_from_basis': f'{basis}+{int(buffer_days)}d'}


def traffic_vintage_usable(survey_year_ce, available_from_utc, target_date, issue_time_utc):
    """Next-stage guard: only a PRIOR survey year whose proxy availability is at or before the issue time."""
    if not available_from_utc:
        return False
    return int(survey_year_ce) < target_date.year and pd.Timestamp(available_from_utc) <= pd.Timestamp(issue_time_utc)


def aadt_unit_flags(columns, patterns):
    return [norm_header(c) for c in columns if any(p.lower() in norm_header(c).lower() for p in patterns)]


AADT_SUMMARY_COLUMNS = ['survey_year_be', 'survey_year_ce', 'province_key', 'province_text_values', 'province_key_status', 'n_rows',
                        'n_station_rows', 'n_duplicate_key_rows', 'n_incomplete_identity_rows', 'n_valid_total', 'n_invalid_total_value',
                        'mean_total_aadt', 'median_total_aadt', 'n_valid_heavy_share', 'n_invalid_heavy_share_value', 'mean_heavy_share_pct',
                        'in_v1_panel', 'unit']


def aadt_province_summary(norm, panel_provinces):
    """Province key x survey year from the output of aadt_station_rows (blank non-records are already excluded). Only row_class
    'station' feeds the statistics; duplicate-key and incomplete-identity rows are counted, never merged. mean/median use valid totals
    (finite, >= 0) and the heavy-share mean valid shares (finite, 0..100). Rows with a blank province cannot be assigned and are counted
    only in the per-year audit. Never a sum (survey points cannot be added into provincial vehicles)."""
    if not len(norm):
        return pd.DataFrame(columns=AADT_SUMMARY_COLUMNS)
    keys = ['survey_year_be', 'survey_year_ce', 'province_key']
    d = norm[norm['province_key'].astype(str).ne('')]
    if not len(d):
        return pd.DataFrame(columns=AADT_SUMMARY_COLUMNS)
    g = d.groupby(keys)
    s = pd.DataFrame({'province_text_values': g['province_text'].agg(lambda v: ';'.join(sorted(set(v)))),
                      'province_key_status': g['province_key_status'].agg(lambda v: ';'.join(sorted(set(v)))), 'n_rows': g.size()})
    for cls, col in (('station', 'n_station_rows'), ('duplicate_key', 'n_duplicate_key_rows'), ('incomplete_identity', 'n_incomplete_identity_rows')):
        s[col] = d['row_class'].eq(cls).groupby([d[k] for k in keys]).sum()
    st = d[d['row_class'].eq('station')]
    st = st.assign(_t=st['total_aadt'].where(st['total_valid']), _h=st['heavy_share_pct'].where(st['heavy_share_valid']))
    agg = st.groupby(keys).agg(n_valid_total=('_t', 'count'), n_invalid_total_value=('total_invalid_value', 'sum'),
                               mean_total_aadt=('_t', 'mean'), median_total_aadt=('_t', 'median'), n_valid_heavy_share=('_h', 'count'),
                               n_invalid_heavy_share_value=('heavy_share_invalid_value', 'sum'), mean_heavy_share_pct=('_h', 'mean'))
    s = s.join(agg).reset_index()
    for c in ('n_station_rows', 'n_duplicate_key_rows', 'n_incomplete_identity_rows', 'n_valid_total', 'n_invalid_total_value',
              'n_valid_heavy_share', 'n_invalid_heavy_share_value'):
        s[c] = s[c].fillna(0).astype(int)
    s['in_v1_panel'] = s['province_key'].isin(panel_provinces) if panel_provinces else None
    s['unit'] = 'vehicles/day at DOH survey points (mean/median across station rows; not a provincial total)'
    return s[AADT_SUMMARY_COLUMNS]


# ---------- reviewed bank-holiday calendar (versioned add/remove, known_from as-of) ----------
def calendar_entries(cal):
    """Flatten reviewed config sources into entry dicts. Raises on bad action/date/year, missing known_from, duplicate entries
    or an add and remove of the same date with the same known_from (ambiguous order)."""
    fields = cal['entry_fields']
    out, seen, by_day = [], set(), {}
    for s in cal['sources']:
        for e in s.get('entries', []):
            r = dict(zip(fields, e))
            d = dt.date.fromisoformat(r['date'])
            if r['action'] not in CAL_ACTIONS:
                raise ValueError(f"{s['id']}: action {r['action']!r} not in {CAL_ACTIONS}")
            if d.year not in s['years']:
                raise ValueError(f"{s['id']}: {d} outside source years {s['years']}")
            if not s.get('known_from'):
                raise ValueError(f"{s['id']}: entries need known_from")
            kf = dt.date.fromisoformat(s['known_from']).isoformat()
            key = (r['date'], r['action'], s['id'])
            if key in seen:
                raise ValueError(f'duplicate calendar entry {key}')
            seen.add(key)
            acts = by_day.setdefault((r['date'], kf), set())
            acts.add(r['action'])
            if len(acts) > 1:
                raise ValueError(f"add and remove of {r['date']} with the same known_from {kf}: order ambiguous")
            out.append({'date': d.isoformat(), 'name': r['name'], 'action': r['action'], 'page_ref': r.get('page') or s.get('page_ref'),
                        'source_id': s['id'], 'known_from': kf})
    return out


def review_source_status(src, reviewed_sha, live_sha, text_dates=(), entry_dates=()):
    """(apply?, reason) for one reviewed source. PDF/PNG: reviewed bytes = the visually reviewed preflight copy; a pinned
    reviewed_sha256 must match it and a live/cached download must be byte-identical, else entries are NOT applied.
    HTML: a snapshot must exist and its text must mention every entry date."""
    if src['kind'] == 'html':
        if not live_sha:
            return False, 'html_snapshot_unavailable'
        missing = sorted(set(entry_dates) - set(text_dates))
        if missing:
            return False, 'html_date_text_not_found:' + ','.join(missing)
        return True, 'html_snapshot_date_text_found'
    if not reviewed_sha:
        if src.get('reviewed_sha256') and live_sha == src['reviewed_sha256']:
            return True, 'live_bytes_match_pinned_reviewed_sha256'
        return False, 'reviewed_copy_missing'
    if src.get('reviewed_sha256') and src['reviewed_sha256'] != reviewed_sha:
        return False, 'reviewed_copy_hash_differs_from_config_pin'
    if live_sha and live_sha != reviewed_sha:
        return False, 'live_bytes_differ_from_reviewed_copy'
    return True, 'live_bytes_identical_to_reviewed_copy' if live_sha else 'reviewed_copy_only_live_unavailable'


def available_from_local(known_from):
    """Date-only known_from is usable from the END of that local day: known_from + 1 day at 00:00 local (naive)."""
    return dt.datetime.combine(dt.date.fromisoformat(str(known_from)) + dt.timedelta(days=1), dt.time(0))


def bank_holidays_as_of(records, issue_local):
    """BOT bank-holiday dates known at naive local datetime issue_local: applied records only, replayed in known_from order
    (adds before removes on the same day). Absence is NOT an ordinary day unless calendar_year_coverage says the year is known."""
    r = records[records['applied'].astype(bool)]
    r = r[r['known_from'].map(available_from_local) <= issue_local]
    r = r.assign(_o=r['action'].map({'add': 0, 'remove': 1})).sort_values(['known_from', '_o', 'date'])
    days = set()
    for d, a in zip(r['date'], r['action']):
        if a == 'add':
            days.add(d)
        else:
            days.discard(d)
    return days


def calendar_year_coverage(records, sources, year, issue_local):
    """Coverage of the reviewed BANK calendar for `year` at issue_local; never claims completeness (later amendments unknown)."""
    base = [s['id'] for s in sources if s.get('role') == 'annual_base' and year in s.get('years', [])]
    ok = records[records['applied'].astype(bool) & records['source_id'].isin(base)]
    if not len(ok):
        return 'unknown_no_applied_reviewed_annual_source'
    if ok['known_from'].map(available_from_local).min() > issue_local:
        return 'unknown_annual_source_not_yet_known'
    return 'reviewed_annual_base_known_later_amendments_unknown'


def festival_rule_evidence(records, recurring, years):
    """Recurring fixed-date festival rules per year with the reviewed bank-holiday dates that coincide (supporting evidence only)."""
    rows = []
    for y in years:
        for f in recurring:
            rule = [dt.date(y, m, d).isoformat() for m, d in f['month_days']]
            hit = records[records['applied'].astype(bool) & records['action'].eq('add') & records['date'].isin(rule)]
            rows.append({'year': y, 'festival_id': f['id'], 'name': f['name'], 'rule_dates': ';'.join(rule),
                         'reviewed_bank_holiday_dates': ';'.join(sorted(set(hit['date']))),
                         'sources': ';'.join(sorted(set(hit['source_id'].astype(str)))),
                         'note': 'deterministic calendar rule; bank-holiday membership (weekend observances shift) is supporting evidence only'})
    return pd.DataFrame(rows, columns=['year', 'festival_id', 'name', 'rule_dates', 'reviewed_bank_holiday_dates', 'sources', 'note'])


# ---------- weather aggregation (overrides the part-1 version: precipitation is a preceding-hour sum) ----------
RAIN_INTERVAL_CONVENTION = 'preceding_hour_sum: value at valid time t covers (t-1h, t]; bucketed by interval start t-1h'


def aggregate_weather(hourly, block_hours, temp_col, rain_col, buffer_hours, issue_hour_local, utc_offset_seconds):
    """Per province x local date x block (24 = daily, 6 = 00-06/06-12/12-18/18-24). Open-Meteo hourly temperature is the INSTANT value
    at valid time; precipitation is the sum over the PRECEDING hour (https://open-meteo.com/en/docs). Temperature is bucketed by valid
    time; precipitation by its interval start (valid time - 1 h), so 00-06 rain sums the values valid 01:00..06:00 and the day sums
    01:00..next-day 00:00. Target keys come from the temperature grid only (the first midnight rain value belongs to the previous day and
    never creates a stray bucket). Each variable needs block_hours rows, unique hours and non-null values, else NaN + missing flag (never a
    partial sum or 0); a last day without its next-midnight value stays rain_missing. issue_bound_ok uses the ORIGINAL nominal issue
    bounds of every row in the bucket (rain's latest valid time is the interval end): latest bound + buffer <= (date - 1) at issue_hour_local.
    Audit only (never targets/predictors): n_*_value_masked counts hours masked by apply_weather_value_policy ('<var>_quality'), and
    *_missing_reason separates value quality (negative_value_masked, null_in_source) from hours not returned (chunk not collected/edge)."""
    keys = ['province_code', 'date', 'block']

    def prep(col):
        d = hourly[['province_code', 'valid_time_local', 'nominal_issue_bound_utc', col]].copy()
        q = hourly[col + '_quality'] if col + '_quality' in hourly else pd.Series('valid', index=hourly.index).where(d[col].notna(), 'null_in_source')
        d['_masked'], d['_null'] = q.eq('negative_value_masked').to_numpy(), q.eq('null_in_source').to_numpy()
        return d
    tv = prep(temp_col)
    tv['date'], tv['block'] = tv['valid_time_local'].dt.normalize(), tv['valid_time_local'].dt.hour // block_hours
    rv = prep(rain_col)
    start = rv['valid_time_local'] - pd.Timedelta(hours=1)
    rv['date'], rv['block'] = start.dt.normalize(), start.dt.hour // block_hours
    t = tv.groupby(keys).agg(n_rows=('valid_time_local', 'size'), n_hours=('valid_time_local', 'nunique'), n_valid_temp=(temp_col, 'count'),
                             _temp=(temp_col, 'mean'), _t_min=('nominal_issue_bound_utc', 'min'), _t_max=('nominal_issue_bound_utc', 'max'),
                             n_temp_value_masked=('_masked', 'sum'), _t_null=('_null', 'sum'))
    r = rv.groupby(keys).agg(n_rain_rows=('valid_time_local', 'size'), n_rain_hours=('valid_time_local', 'nunique'),
                             n_valid_rain=(rain_col, 'count'), _rain=(rain_col, 'sum'), _r_min=('nominal_issue_bound_utc', 'min'),
                             _r_max=('nominal_issue_bound_utc', 'max'), rain_latest_valid_time_local=('valid_time_local', 'max'),
                             n_rain_value_masked=('_masked', 'sum'), _r_null=('_null', 'sum'))
    a = t.join(r, how='left').reset_index()
    for c in ('n_rain_rows', 'n_rain_hours', 'n_valid_rain', 'n_rain_value_masked', '_r_null', 'n_temp_value_masked', '_t_null'):
        a[c] = a[c].fillna(0).astype(int)
    a['n_expected'] = block_hours
    a['temp_missing'] = ~((a['n_rows'] == block_hours) & (a['n_hours'] == block_hours) & (a['n_valid_temp'] == block_hours))
    a['rain_missing'] = ~((a['n_rain_rows'] == block_hours) & (a['n_rain_hours'] == block_hours) & (a['n_valid_rain'] == block_hours))
    a['rain_sum'] = a['_rain'].where(~a['rain_missing'])
    a['temp_mean'] = a['_temp'].where(~a['temp_missing'])
    for v, rows, hours, masked, null in (('rain', 'n_rain_rows', 'n_rain_hours', 'n_rain_value_masked', '_r_null'),
                                         ('temp', 'n_rows', 'n_hours', 'n_temp_value_masked', '_t_null')):
        a[f'{v}_missing_reason'] = [';'.join(s for s, c in (('negative_value_masked', m), ('null_in_source', z),
                                                             ('hours_not_returned', n != block_hours or u != block_hours)) if c) if miss else ''
                                    for miss, m, z, n, u in zip(a[f'{v}_missing'], a[masked], a[null], a[rows], a[hours])]
    a['earliest_issue_bound_utc'] = a[['_t_min', '_r_min']].min(axis=1)
    a['latest_issue_bound_utc'] = a[['_t_max', '_r_max']].max(axis=1)
    a = a.drop(columns=['_rain', '_temp', '_t_min', '_t_max', '_r_min', '_r_max', '_r_null', '_t_null'])
    a['rain_interval_convention'] = RAIN_INTERVAL_CONVENTION
    a['time_block'] = 'day' if block_hours == 24 else a['block'].map(lambda b: f'{b * block_hours:02d}-{(b + 1) * block_hours:02d}')
    issue_local = a['date'] - pd.Timedelta(days=1) + pd.Timedelta(hours=issue_hour_local)
    a['issue_time_utc'] = issue_local - pd.Timedelta(seconds=utc_offset_seconds)
    a['issue_bound_ok'] = (a['latest_issue_bound_utc'] + pd.Timedelta(hours=buffer_hours)) <= a['issue_time_utc']
    return a


# ---------- deterministic derived tables ----------
def write_csv_gz(path, df):
    """Gzip CSV with a fixed header (mtime 0, no stored file name): identical tables give identical bytes on every rerun."""
    import gzip
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with open(tmp, 'wb') as fh, gzip.GzipFile(filename='', mode='wb', fileobj=fh, mtime=0) as gz:
        gz.write(df.to_csv(index=False).encode('utf-8'))
    os.replace(tmp, path)
    return path


# ---------- pinned local-first sources (geography) ----------
def entry_file(entry, root):
    """Local file of a registry entry: a notebook download lives at root/path; a registered Codex preflight copy has path None and
    lives at preflight_path (it was never downloaded by this notebook). None when the entry has no usable bytes."""
    if not entry or not entry.get('sha256'):
        return None
    if entry.get('path'):
        return Path(root) / entry['path']
    if entry.get('preflight_path'):
        return Path(entry['preflight_path'])
    return None


def pinned_source(key, spec, preflight_dir, root, registry, fetch_fn, refresh=False, **fetch_kw):
    """Exact sha256-pinned source, local first. Returns (entry, file, info).
    1. A Codex preflight copy is registered read-only (register_preflight refuses changed bytes) and used without any network call
       when its sha256 equals spec['expected_sha256']; provenance keeps the cache mtime and an unknown original retrieval time.
    2. Without a preflight copy, fetch_fn (hash-verified registry cache, else HTTPS download registered as downloaded_by_notebook09).
    3. refresh=True also re-fetches the remote even when the preflight copy is valid; changed remote bytes stay a new registry
       version and are never used or relabelled (info['remote_matches_pin'] False -> the caller records a review gap).
    Bytes that differ from the pin are refused: CacheIntegrityError when spec['required'], else (None, None, info) with the reason."""
    exp = spec['expected_sha256']
    info = {'key': key, 'url': spec['url'], 'expected_sha256': exp, 'used': None, 'used_sha256': None, 'remote_checked': False,
            'remote_sha256': None, 'remote_matches_pin': None, 'error': None}
    used, path = None, None
    pre = Path(preflight_dir) / spec['file']
    if pre.is_file():
        pe = register_preflight('preflight:' + key, spec['url'], pre, registry, 'codex_preflight_download_original_time_unknown',
                                meta={'expected_sha256': exp, 'source_group': 'geography'})
        if pe['sha256'] != exp:
            msg = f"{key}: preflight copy {pre} sha256 {pe['sha256'][:12]} != configured pin {exp[:12]}; review it and change the pin explicitly"
            if spec.get('required'):
                raise CacheIntegrityError(msg)
            info['error'] = msg
        else:
            used, path, info['used'] = pe, pre, 'codex_preflight_registered_pin_verified'
    if used is None or refresh:
        e = fetch_fn(spec['resource_id'], spec['url'], spec['rel'], meta={'expected_sha256': exp, 'source_group': 'geography'}, **fetch_kw)
        info['remote_checked'] = e.get('last_status') not in (None, 'cache_hit_hash_verified', 'unavailable')  # remote bytes received this run
        info['remote_sha256'] = e.get('sha256')
        info['remote_matches_pin'] = (e['sha256'] == exp) if e.get('sha256') else None
        if used is None:
            if not e.get('sha256'):
                info['error'] = info['error'] or f"{key}: no preflight copy and no download/cache: {e.get('error')}"
            elif e['sha256'] != exp:
                msg = (f"{key}: notebook copy {e['path']} sha256 {e['sha256'][:12]} != configured pin {exp[:12]} (bytes kept in the registry, "
                       'not used); review the change and update the pin explicitly')
                if spec.get('required'):
                    raise CacheIntegrityError(msg)
                info['error'] = msg
            else:
                used, path = e, entry_file(e, root)
                info['used'] = 'notebook_cache_pin_verified' if e.get('last_status') == 'cache_hit_hash_verified' else 'notebook_download_pin_verified'
    if used is not None:
        info.update(used_sha256=used['sha256'], provenance=used.get('provenance'), cache_mtime_utc=used.get('cache_mtime_utc'),
                    retrieved_at_utc=used.get('retrieved_at_utc'), verified_at_utc=used.get('verified_at_utc'),
                    original_download_time=used.get('original_download_time'))
    return used, path, info


# ---------- audited province keys (exact alias map only) ----------
PROVINCE_KEY_STATUSES = ('blank', 'v1_panel_exact', 'v1_panel_audited_alias', 'not_in_v1_panel', 'v1_panel_unavailable')


def province_keys(values, aliases, panel):
    """Raw province text -> v1 panel join key via an EXACT audited full-string alias map; no fuzzy, prefix or substring rules.
    Returns province_raw (unchanged), province_text (whitespace-normalized), province_key and province_key_status."""
    raw = pd.Series(list(values), dtype=object)
    text = raw.map(norm_text)
    key = text.map(lambda t: aliases.get(t, t))
    panel = set(panel or [])

    def status(t, k):
        if not t:
            return 'blank'
        if not panel:
            return 'v1_panel_unavailable'
        if t in panel:
            return 'v1_panel_exact'
        return 'v1_panel_audited_alias' if t in aliases and k in panel else 'not_in_v1_panel'
    return pd.DataFrame({'province_raw': raw, 'province_text': text, 'province_key': key,
                         'province_key_status': [status(t, k) for t, k in zip(text, key)]})


# ---------- AADT row classes and value validity ----------
AADT_ROW_CLASSES = ('blank_nonrecord', 'incomplete_identity', 'duplicate_key', 'station')
AADT_KEY = ['route', 'control_section', 'survey_point']


def aadt_station_rows(raw, norm, rep, aliases, panel):
    """Classify every parsed CSV row and flag invalid values. Returns (rows without blank non-records, audit dict).
    blank_nonrecord: every raw cell blank (a filler CSV line, not a survey record; counted, excluded from station data/statistics).
    incomplete_identity: some content but a blank route/control_section/survey_point/province (uncertain; never merged or guessed).
    duplicate_key: complete key shared with another complete row (flagged; excluded from statistics).
    station: everything else. n_parsed == sum of the four classes. Values: total valid when finite and >= 0; heavy share valid when
    finite and within 0..100; other present values are invalid (counted, excluded from statistics, never used as features)."""
    def is_blank(df):  # plain numpy bool matrix (string dtypes would give nullable booleans)
        return np.column_stack([df[c].astype(str).str.strip().eq('').fillna(True).to_numpy(dtype=bool) for c in df.columns])
    blank = is_blank(raw).all(axis=1)
    incomplete = ~blank & is_blank(norm[AADT_KEY + ['province']]).any(axis=1)
    complete = ~blank & ~incomplete
    dup = np.zeros(len(norm), dtype=bool)
    if complete.any():
        dup[complete] = norm.loc[complete, AADT_KEY].astype(str).duplicated(keep=False).to_numpy(dtype=bool)
    out = norm.copy()
    out.insert(0, 'csv_data_row_index', np.arange(len(norm)))
    cls = pd.Series('station', index=norm.index, dtype=object)
    cls[dup], cls[incomplete], cls[blank] = 'duplicate_key', 'incomplete_identity', 'blank_nonrecord'
    out['row_class'] = cls
    pk = province_keys(raw[rep['mapped']['province']] if 'province' in rep['mapped'] else norm['province'], aliases, panel)
    pk.index = out.index
    out['province'] = pk['province_text']
    for c in pk.columns:
        out[c] = pk[c]
    tot = pd.to_numeric(out['total_aadt'], errors='coerce').astype(float)
    hv = pd.to_numeric(out['heavy_share_pct'], errors='coerce').astype(float)
    out['total_valid'] = np.isfinite(tot) & (tot >= 0)
    out['total_invalid_value'] = tot.notna() & ~out['total_valid']
    out['heavy_share_valid'] = np.isfinite(hv) & (hv >= 0) & (hv <= 100)
    out['heavy_share_invalid_value'] = hv.notna() & ~out['heavy_share_valid']
    cnt = out[[c for c in out.columns if c.startswith('count__')]].astype(float)
    out['count_invalid_value'] = (cnt.notna() & (~np.isfinite(cnt) | (cnt < 0))).any(axis=1) if cnt.shape[1] else False
    counts = out['row_class'].value_counts()
    nb = out[~blank]
    st = nb[nb['row_class'].eq('station')]
    audit = {'rows_parsed': len(out), **{f'rows_{c}': int(counts.get(c, 0)) for c in AADT_ROW_CLASSES},
             'station_valid_total_rows': int(st['total_valid'].sum()), 'station_valid_heavy_share_rows': int(st['heavy_share_valid'].sum()),
             'nonblank_invalid_total_value_rows': int(nb['total_invalid_value'].sum()),
             'nonblank_invalid_heavy_share_value_rows': int(nb['heavy_share_invalid_value'].sum()),
             'nonblank_invalid_count_value_rows': int(nb['count_invalid_value'].sum()),
             'nonblank_missing_total_rows': int(nb['total_aadt'].isna().sum()),
             'province_key_status_counts': {k: int(v) for k, v in nb['province_key_status'].value_counts().sort_index().items()},
             'province_alias_values': {t: aliases[t] for t in sorted(set(nb.loc[nb['province_key_status'].eq('v1_panel_audited_alias'), 'province_text']))},
             'provinces_not_in_v1_panel': sorted(set(nb.loc[nb['province_key_status'].eq('not_in_v1_panel'), 'province_text']))}
    assert audit['rows_parsed'] == sum(audit[f'rows_{c}'] for c in AADT_ROW_CLASSES)
    return nb.reset_index(drop=True), audit

## ตั้งค่า (paths/config/flags)

หา backend จาก `ROADAWARE_BACKEND_ROOT` หรือโฟลเดอร์ที่เปิดอยู่ (root หรือ notebooks ก็ได้), อ่าน config, กำหนด external root / report root / v1 data root (อ่านอย่างเดียว) / preflight และโหมด offline/refresh; ปฏิเสธถ้า output ทับตำแหน่งของ v1

In [3]:
# SETUP: roots, config and run flags (env overrides are set by backend/scripts/run_v2_notebooks.py; VS Code Run All uses defaults)
import sys


def find_backend():
    """ROADAWARE_BACKEND_ROOT, else the first cwd ancestor that is (or contains) backend/ with config + notebooks."""
    env = os.environ.get('ROADAWARE_BACKEND_ROOT')
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        for cand in (d, d / 'backend'):
            if (cand / 'config' / 'external_sources_v2.json').is_file() and (cand / 'notebooks').is_dir():
                return cand
    raise FileNotFoundError('backend root not found from the working directory; set ROADAWARE_BACKEND_ROOT')


def env_flag(name):
    return os.environ.get(name, '').strip().lower() in ('1', 'true', 'yes')


def env_path(name, default):
    v = os.environ.get(name)
    return Path(v).expanduser().resolve() if v else Path(default).resolve()


def inside(p, root):
    p, root = Path(p).resolve(), Path(root).resolve()
    return p == root or root in p.parents


BACKEND = find_backend()
CFG_PATH = BACKEND / 'config' / 'external_sources_v2.json'
CFG = json.loads(CFG_PATH.read_text('utf-8'))
EXT = env_path('ROADAWARE_V2_EXTERNAL_ROOT', BACKEND / CFG['default_external_root'])
OUT = env_path('ROADAWARE_V2_OUTPUT_ROOT', BACKEND)
V1 = env_path('ROADAWARE_V1_DATA_ROOT', BACKEND)
PREFLIGHT = env_path('ROADAWARE_V2_PREFLIGHT_DIR', BACKEND / CFG['preflight_dir'])
REP = OUT / CFG['report_subdir']
REGISTRY_PATH = EXT / 'registry.json'
OFFLINE = env_flag('ROADAWARE_V2_OFFLINE')
REFRESH = env_flag('ROADAWARE_V2_REFRESH')
if OFFLINE and REFRESH:
    raise ValueError('ROADAWARE_V2_OFFLINE and ROADAWARE_V2_REFRESH are mutually exclusive')
V1_PROTECTED = [V1 / 'data' / 'raw', V1 / 'data' / 'interim', V1 / 'data' / 'processed', V1 / 'data' / 'model', V1 / 'reports' / 'data',
                V1 / 'reports' / 'models']
for _name, _p in (('external root', EXT), ('report dir', REP)):
    _hit = [str(v) for v in V1_PROTECTED if inside(_p, v) or inside(v, _p)]
    if _hit:
        raise ValueError(f'{_name} {_p} overlaps protected v1 locations {_hit}; v1 artifacts must stay untouched')
if not (V1 / 'data' / 'raw' / 'accidents' / 'source_manifest.json').is_file():
    raise FileNotFoundError(f'v1 raw snapshot manifest not found under {V1}; set ROADAWARE_V1_DATA_ROOT')


def protected_tree_hashes(roots, base):
    """sha256 of every file under the protected v1 locations (relative path -> hash); compared again in the final checks."""
    return {p.relative_to(base).as_posix(): sha256_file(p) for r in roots if r.is_dir() for p in sorted(r.rglob('*')) if p.is_file()}


V1_TREE_BEFORE = protected_tree_hashes(V1_PROTECTED, V1)
EXT.mkdir(parents=True, exist_ok=True)
REP.mkdir(parents=True, exist_ok=True)
pd.set_option('display.width', 200)
print(f'backend={BACKEND}\nexternal_root={EXT}\nreports={REP}\nv1_data_root={V1} (read-only)\npreflight={PREFLIGHT} (read-only)\n'
      f'offline={OFFLINE} refresh={REFRESH} python={sys.executable}')

backend=/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend
external_root=/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/data/external_v2
reports=/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/reports/v2/sources
v1_data_root=/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend (read-only)
preflight=/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/backend/data/external_v2/evidence (read-only)
offline=False refresh=False python=/Users/earthnaja/Documents/GitHub/Work - Earth/Project Work/Source Code/RoadAware/.venv/bin/python


## A. Registry, งบคำขอเครือข่าย, อินพุต v1 (อ่านอย่างเดียว) และ metadata ของ catalog ทางการ

- `registry.json` ใต้ external root เก็บ URL, URL ที่ resolve, เวลาดึง/ตรวจ, SHA256, ขนาด, header, สถานะ ของทุกไฟล์; ไฟล์ preflight ของ Codex ลงทะเบียนแยก (`preflight:*`) พร้อม mtime ของแคช และระบุว่าเวลาดาวน์โหลดเดิม **ไม่ทราบ**
- ดึง `package_show` ปัจจุบันของ MOT (roadaccident) และกรมทางหลวง (AADT) เก็บเป็น snapshot ที่ลงทะเบียน hash; ถ้าดึงไม่ได้จึงใช้สำเนา preflight พร้อมบันทึกเป็นช่องว่าง
- ตรวจ SHA256 ของ snapshot ดิบ v1 ทั้ง 6 ไฟล์ก่อนใช้ และตรวจซ้ำตอนจบว่าไม่เปลี่ยน

In [4]:
# A. Registry, request budget, read-only v1 inputs and official catalog metadata (MOT accidents, DOH traffic)
NET = CFG['http']
BUDGET = RequestBudget(NET['max_network_requests_per_run'])
REG = load_registry(REGISTRY_PATH)
RUN = {'started_utc': utc_now(), 'sections': {}}
GAPS, COVERAGE, DERIVED = [], [], []
(EXT / 'derived').mkdir(parents=True, exist_ok=True)


def save_reg():
    write_json(REGISTRY_PATH, REG)


def fetch(res_id, url, rel, **kw):
    e = try_fetch(res_id, url, rel, EXT, REG, NET, BUDGET, refresh=REFRESH, offline=OFFLINE, **kw)
    save_reg()
    return e


V1MAN = json.loads((V1 / 'data' / 'raw' / 'accidents' / 'source_manifest.json').read_text('utf-8'))
V1_HASHES_BEFORE = verify_v1_snapshot(V1, V1MAN)
panel_path = V1 / 'data' / 'processed' / 'province_day_panel.csv'
PANEL_PROVINCES = (sorted(pd.read_csv(panel_path, usecols=['province'], dtype=str)['province'].dropna().str.strip().unique().tolist())
                   if panel_path.is_file() else [])
if not PANEL_PROVINCES:
    GAPS.append('v1 province_day_panel.csv not found; province names were not compared with the v1 panel')


def catalog(key, res_id):
    """Live package_show snapshot (cached, hash-registered); falls back to the Codex preflight copy only if the live fetch fails."""
    pf = CFG['preflight_files'][key]
    pre = None
    if (PREFLIGHT / pf['file']).is_file():
        pre = register_preflight('preflight:' + key, pf['url'], PREFLIGHT / pf['file'], REG, 'codex_preflight_download_original_time_unknown')
    live = fetch(res_id, pf['url'], f'raw/catalog/{key}.json', validate=json_validator(require_success=True))
    if live.get('sha256'):
        return json.loads((EXT / live['path']).read_text('utf-8')), 'notebook_snapshot', live, pre
    if pre:
        GAPS.append(f'{key}: live package_show unavailable ({live.get("error")}); using the Codex preflight copy')
        return json.loads((PREFLIGHT / pf['file']).read_text('utf-8')), 'codex_preflight', pre, pre
    raise RuntimeError(f'{key}: neither a live catalog snapshot nor the preflight copy is available; check network/allowlist')


MOT_PKG, mot_src, mot_e, mot_pre = catalog('mot_package', 'mot_package_show')
DOH_PKG, doh_src, doh_e, doh_pre = catalog('doh_package', 'doh_package_show')
for key, pkg, src, e, pre in (('mot_package', MOT_PKG, mot_src, mot_e, mot_pre), ('doh_package', DOH_PKG, doh_src, doh_e, doh_pre)):
    r = pkg['result']
    RUN['sections'][key] = {'used': src, 'sha256': e.get('sha256'), 'retrieved_at_utc': e.get('retrieved_at_utc'),
                            'preflight_sha256': (pre or {}).get('sha256'), 'preflight_identical': bool(pre) and pre.get('sha256') == e.get('sha256'),
                            'metadata_modified': r.get('metadata_modified'), 'license': r.get('license_title'), 'n_resources': len(r.get('resources', []))}
    COVERAGE.append({'source_group': 'catalog', 'resource_id': key, 'year': None, 'format': 'ckan_package_show_json',
                     'rows': len(r.get('resources', [])), 'n_columns': None, 'mapped_fields': 'resources[url,format,created,last_modified,size]',
                     'missing_fields': None, 'unmapped_headers': None, 'coverage_start': None, 'coverage_end': None,
                     'status': src, 'sha256': e.get('sha256'), 'note': f"license {r.get('license_title')}; metadata_modified {r.get('metadata_modified')}"})
print('v1 raw snapshot verified (read-only):', len(V1_HASHES_BEFORE), 'files | panel provinces:', len(PANEL_PROVINCES))
print({k: RUN['sections'][k] for k in ('mot_package', 'doh_package')})

v1 raw snapshot verified (read-only): 6 files | panel provinces: 77
{'mot_package': {'used': 'notebook_snapshot', 'sha256': '1b02b03a8715e50c35c9c6afed0deac85a959f288074d2cbc2f5dee72278a83e', 'retrieved_at_utc': '2026-11-10T05:58:13+00:00', 'preflight_sha256': '1b02b03a8715e50c35c9c6afed0deac85a959f288074d2cbc2f5dee72278a83e', 'preflight_identical': True, 'metadata_modified': '2026-11-02T03:23:46.021163', 'license': 'Open Data Common', 'n_resources': 16}, 'doh_package': {'used': 'notebook_snapshot', 'sha256': 'c7eb5182487a14647a97b39ba79ec30da54d5c4ea29c58cbedd0d091c0269ec9', 'retrieved_at_utc': '2026-11-10T05:58:34+00:00', 'preflight_sha256': 'c7eb5182487a14647a97b39ba79ec30da54d5c4ea29c58cbedd0d091c0269ec9', 'preflight_identical': True, 'metadata_modified': '2026-03-11T03:35:24.201407', 'license': 'License not specified', 'n_resources': 18}}


## B. หลักฐานวันที่รายปี (Date evidence)

ปัญหาเดิม: ไฟล์ v1 ปี 2021–2023 มีวันที่แบบ `xx/xx/yyyy` ที่อ่านได้ทั้ง D/M และ M/D (v1 กักกันไว้ ทำให้วันที่ 1–12 ของเดือนถูกนับขาด), ปี 2024/2025 เป็น Excel serial, ปี 2026 ไม่มี ACC_CODE

วิธี: อ่าน XLSX ทางการของ MOT (preflight ของ Codex ถ้าเป็นไฟล์ XLSX สมบูรณ์ มิฉะนั้นดาวน์โหลดจาก URL ใน package) เฉพาะเซลล์วันที่ที่เป็นชนิด **date/datetime จริง** แล้วจับคู่กับแถว CSV v1 ด้วย ACC_CODE เดิมแบบตรงตัว (ไม่ fuzzy ไม่อาศัยลำดับแถว) แถวหนึ่ง resolve ได้เมื่อ: ทุกแถว XLSX ของ ID นั้นมีวันที่ typed เดียวกัน, ทุกแถว CSV ของ ID นั้นมีสตริงวันที่เดียวกัน และวันที่ XLSX เป็นหนึ่งในการอ่านที่เป็นไปได้ของแถว CSV นั้นเอง

- ข้อสรุปเป็น **รายปี** (ไม่ใช้กฎเดียวทุกปี) และใช้เพื่อรายงานเท่านั้น การ resolve ทำรายแถว
- ID ซ้ำ/ขัดกัน, ID ที่ไม่มีใน XLSX, เซลล์ที่เป็นข้อความ, วันที่ไม่ตรง → ไม่ resolve (คงการกักกันแบบ v1) พร้อมนับจำนวน
- ID ที่มีเฉพาะใน XLSX (แถวใหม่/แก้ไข) นับไว้แต่ **ไม่เพิ่ม** เป็นเหตุการณ์ และไม่แทนที่ไฟล์ดิบ v1
- ปี 2026 ไม่มี ID จึงทำเพียงโปรไฟล์รูปแบบวันที่ (diagnostic)

In [5]:
# B. Date evidence by year: v1 raw CSV readings vs TYPED XLSX date cells, exact original ACC_CODE join + supporting context
D = CFG['date_evidence']
CTX = D['context_fields']
mot_res = {r['url'].rsplit('/', 1)[-1]: r for r in MOT_PKG['result']['resources'] if r.get('url')}
DATE_SUMMARY, row_frames = [], []
for f in V1MAN['files']:
    ce, be = int(f['ce_year']), int(f['be_year'])
    if ce not in D['years_ce']:
        continue
    csv = pd.read_csv(V1 / f['path'], dtype=str, keep_default_na=False, encoding=f['encoding'])
    assert len(csv) == f['rows'], f['file']
    rec = {'year_ce': ce, 'year_be': be, 'csv_file': f['file'], 'csv_sha256': f['sha256'], 'csv_rows': len(csv),
           'csv_has_acc_code': D['id_header'] in csv.columns, **date_format_profile(csv[D['date_header']], ce)}
    if ce in D['diagnostic_only_years_ce']:
        rec['use_policy'] = 'diagnostic_only_no_acc_code_not_a_label'
    elif ce in D['xlsx_years_ce']:
        name = D['xlsx_name_pattern'].format(year=ce)
        res = mot_res.get(name)
        url = res['url'] if res else None
        xpath, xsrc, xe = None, None, {}
        pre = PREFLIGHT / name
        if pre.is_file():
            xe = register_preflight(f'preflight:mot_accident_{ce}_xlsx', url, pre, REG, 'codex_preflight_download_original_time_unknown',
                                    meta={'catalog': resource_meta(res) if res else None})
            if zipfile.is_zipfile(pre):
                xpath, xsrc = pre, 'codex_preflight'
            else:
                xe['last_status'] = 'preflight_not_a_complete_xlsx'
                GAPS.append(f'date evidence: preflight {name} is not a complete XLSX (download may have been interrupted); trying the official URL')
        if xpath is None and url:
            xe = fetch(f'mot_accident_{ce}_xlsx', url, f'raw/mot/{name}', validate=xlsx_validator, max_bytes=D['max_xlsx_bytes'],
                       meta={'source_group': 'date_evidence', 'catalog': resource_meta(res)})
            if xe.get('sha256'):
                xpath, xsrc = EXT / xe['path'], 'downloaded_by_notebook09'
        save_reg()
        rec.update(xlsx_name=name, xlsx_url=url, xlsx_source=xsrc, xlsx_sha256=xe.get('sha256'), xlsx_bytes=xe.get('bytes'),
                   xlsx_catalog_size=(res or {}).get('size'), xlsx_catalog_last_modified=(res or {}).get('last_modified'))
        rec['xlsx_size_matches_catalog'] = (xe.get('bytes') == res.get('size')) if res and res.get('size') and xe.get('bytes') else None
        if xpath is None:
            rec['use_policy'] = 'xlsx_unavailable_v1_quarantine_kept'
            GAPS.append(f"date evidence: {name} unavailable ({xe.get('error') or 'not listed in MOT package'}); {ce} ambiguous rows stay quarantined")
        else:
            try:
                x, xmeta = read_xlsx_id_dates(xpath, D['id_header'], D['date_header'], D['max_header_scan_rows'])
                xc, xcmeta = read_xlsx_context(xpath, D['id_header'], D['date_header'], CTX, D['max_header_scan_rows'])
            except Exception as ex:  # a corrupt/unexpected workbook is a recorded gap, never a guess
                x, xmeta, xcmeta = None, {'error': f'{type(ex).__name__}: {ex}'}, {}
            rec['xlsx_meta'] = xmeta
            if x is None:
                rec['use_policy'] = 'xlsx_unreadable_v1_quarantine_kept'
                GAPS.append(f"date evidence: {name} unreadable: {xmeta['error']}")
            else:
                csv_rows = pd.DataFrame({'source_row': np.arange(2, len(csv) + 2), 'acc_code': csv[D['id_header']], 'date_raw': csv[D['date_header']]})
                csv_ctx = pd.DataFrame({'source_row': csv_rows['source_row']})
                for fd in CTX:
                    csv_ctx[fd['field']] = csv[fd['header']].map(lambda v, k=fd['kind']: norm_context(v, k)) if fd['header'] in csv.columns else None
                rows, summ = audit_dates_by_id(csv_rows, x, ce)
                rows, ctx_counts = apply_context_check(rows, csv_ctx, xc, CTX)
                ok = rows['resolution_status'] == 'resolved'
                amb = rows['ambiguous_in_v1']
                summ.update(n_resolved_id_date_only=summ['n_resolved'], n_resolved=int(ok.sum()),
                            n_ambiguous_resolved=int((amb & ok).sum()), n_ambiguous_unresolved=int((amb & ~ok).sum()),
                            status_counts=rows['resolution_status'].value_counts().to_dict(),
                            reading_counts=rows.loc[ok, 'xlsx_reading'].value_counts().to_dict(),
                            context_counts=ctx_counts, context_meta=xcmeta,
                            context_missing_csv_headers=[fd['header'] for fd in CTX if fd['header'] not in csv.columns],
                            **candidate_rule_agreement(csv_rows, x, ce))
                if xcmeta.get('missing_headers') or summ['context_missing_csv_headers']:
                    GAPS.append(f"date evidence {ce}: context headers missing (xlsx {xcmeta.get('missing_headers')}, csv "
                                f"{summ['context_missing_csv_headers']}); affected rows stay context_unverified")
                rows.insert(0, 'source_year_be', be)
                rows.insert(0, 'source_year_ce', ce)
                row_frames.append(rows)
                rec.update(summ)
                rec['use_policy'] = 'resolve_only_rows_with_unique_typed_id_date_and_context_agreement'
    else:
        rec['use_policy'] = ('csv_dates_unambiguous_no_resolution_needed' if rec['csv_ambiguous_slash_rows'] == 0
                             else 'ambiguous_rows_without_xlsx_evidence_kept_quarantined')
    DATE_SUMMARY.append(rec)
    print(f"  {ce}: {rec['csv_format_summary']} | policy={rec['use_policy']} | resolved={rec.get('n_resolved', '-')} "
          f"ambiguous_resolved={rec.get('n_ambiguous_resolved', '-')}/{rec['csv_ambiguous_slash_rows']} mismatch={rec.get('n_mismatch', '-')} "
          f"M/D agree={rec.get('rule_mdy_agree', '-')}/{rec.get('rule_compared', '-')} D/M agree={rec.get('rule_dmy_agree', '-')} "
          f"context={rec.get('context_counts', '-')}", flush=True)
DATE_ROWS = pd.concat(row_frames, ignore_index=True) if row_frames else pd.DataFrame(columns=['source_year_ce', 'resolution_status', 'resolved_date'])
DATE_ROWS.to_csv(EXT / 'derived' / 'date_resolution_by_id.csv', index=False)
DERIVED.append('derived/date_resolution_by_id.csv')

# MOT data dictionary: snapshot any dictionary resource listed in the live package; never treated as date-format proof
dd = CFG['mot']['data_dictionary']
dd_urls = [(r.get('name'), r['url']) for r in find_dictionary_resources(MOT_PKG, dd['resource_keywords'])][:3]
if dd.get('url'):
    dd_urls.append(('configured', dd['url']))
DD_ATTEMPTS = []
for i, (nm, u) in enumerate(dd_urls):
    de = fetch(f'mot_data_dictionary_{i}', u, f'raw/mot/data_dictionary_{i}' + (Path(urlparse(u).path).suffix or '.bin'), validate=nonempty_validator)
    DD_ATTEMPTS.append({'name': nm, 'url': u, 'status': de.get('last_status'), 'sha256': de.get('sha256'), 'error': de.get('error')})
dd_status = ('; '.join(f"{a['url']}: {a['status']}" + (f" ({a['error'][:120]})" if a['error'] else '') for a in DD_ATTEMPTS)
             if DD_ATTEMPTS else 'no data-dictionary resource listed in the current MOT package (nothing fetched)')
GAPS.append(f"MOT data dictionary: {dd_status}. {dd['status_note']}")
write_json(REP / 'date_evidence_by_year.json', {'policy': D['policy'], 'data_dictionary_status': dd_status, 'data_dictionary_attempts': DD_ATTEMPTS,
                                                 'per_row_file': 'derived/date_resolution_by_id.csv (under the external root; not in git)',
                                                 'per_row_sha256': sha256_file(EXT / 'derived' / 'date_resolution_by_id.csv'), 'years': DATE_SUMMARY})
write_csv(REP / 'date_evidence_by_year.csv', flatten_records(DATE_SUMMARY))
RUN['sections']['date_evidence'] = {'data_dictionary_status': dd_status, 'years': [r['year_ce'] for r in DATE_SUMMARY]}
_ds = pd.DataFrame(DATE_SUMMARY)
_ds[['year_ce', 'csv_rows', 'csv_ambiguous_slash_rows', 'use_policy'] +
    [c for c in ('n_resolved', 'n_ambiguous_resolved', 'n_mismatch', 'rule_mdy_agree', 'rule_dmy_agree', 'rule_compared') if c in _ds]]

  2021: slash:20062 | policy=resolve_only_rows_with_unique_typed_id_date_and_context_agreement | resolved=20062 ambiguous_resolved=7413/7413 mismatch=0 M/D agree=20062/20062 D/M agree=898 context={'agree': 20062}


  2022: slash:20566 | policy=resolve_only_rows_with_unique_typed_id_date_and_context_agreement | resolved=8135 ambiguous_resolved=7268/7268 mismatch=0 M/D agree=867/8135 D/M agree=8135 context={'not_evaluated': 12431, 'agree': 8135}


  2023: slash:23841 | policy=resolve_only_rows_with_unique_typed_id_date_and_context_agreement | resolved=22922 ambiguous_resolved=7376/8295 mismatch=0 M/D agree=22922/22922 D/M agree=968 context={'agree': 22922, 'not_evaluated': 919}


  2024: excel_serial:23952 | policy=csv_dates_unambiguous_no_resolution_needed | resolved=- ambiguous_resolved=-/0 mismatch=- M/D agree=-/- D/M agree=- context=-


  2025: excel_serial:23715 | policy=csv_dates_unambiguous_no_resolution_needed | resolved=- ambiguous_resolved=-/0 mismatch=- M/D agree=-/- D/M agree=- context=-


  2026: slash:15239 | policy=diagnostic_only_no_acc_code_not_a_label | resolved=- ambiguous_resolved=-/6034 mismatch=- M/D agree=-/- D/M agree=- context=-


,year_ce,csv_rows,csv_ambiguous_slash_rows,use_policy,n_resolved,n_ambiguous_resolved,n_mismatch,rule_mdy_agree,rule_dmy_agree,rule_compared
0,2021,20062,7413,resolve_only_rows_with_unique_typed_id_date_an...,20062.0,7413.0,0.0,20062.0,898.0,20062.0
1,2022,20566,7268,resolve_only_rows_with_unique_typed_id_date_an...,8135.0,7268.0,0.0,867.0,8135.0,8135.0
2,2023,23841,8295,resolve_only_rows_with_unique_typed_id_date_an...,22922.0,7376.0,0.0,22922.0,968.0,22922.0
3,2024,23952,0,csv_dates_unambiguous_no_resolution_needed,NaN,NaN,NaN,NaN,NaN,NaN
4,2025,23715,0,csv_dates_unambiguous_no_resolution_needed,NaN,NaN,NaN,NaN,NaN,NaN
5,2026,15239,6034,diagnostic_only_no_acc_code_not_a_label,NaN,NaN,NaN,NaN,NaN,NaN


## C. ปริมาณจราจร (AADT กรมทางหลวง)

- เลือก resource CSV รายปีสำรวจ พ.ศ. 2563–2568 จาก package ทางการของกรมทางหลวง (ถ้ามีหลายไฟล์ในปีเดียวกันเลือกตัวที่ `created` ล่าสุดและบันทึกตัวอื่นไว้) เก็บไฟล์ดิบไว้ตามเดิม แล้ว normalize เป็น ปี/จังหวัด/สายทาง/ตอนควบคุม/จุดสำรวจ/จำนวนรถแต่ละประเภท/รวม/สัดส่วนรถหนัก
- ตรวจ schema รายปี, ค่าตัวเลข (ไม่เติม 0), key ซ้ำ (สายทาง+ตอนควบคุม+จุดสำรวจ), ผลรวมประเภทรถเทียบคอลัมน์ "รวม" และสัดส่วนรถหนัก
- หน่วย: คัน/วัน ณ จุดสำรวจ **รวมกันเป็นจำนวนรถจริงหรือจราจร real-time ไม่ได้** ขั้นถัดไปใช้ได้เพียง mean/median/coverage รายจังหวัด
- บันทึก metadata ของแต่ละ resource (created/last_modified/metadata_modified/release) เพื่อใช้ตัดสินการ join: โมเดลใช้ได้เฉพาะปีสำรวจที่เผยแพร่ก่อนเวลาออกพยากรณ์ (timestamp ของ catalog เป็นเพียง proxy) และห้ามใช้ค่าปีเป้าหมายเองเป็น feature ต้นปี
- เทียบไฟล์ preflight `aadt_2021.csv` กับไฟล์ที่ notebook ดาวน์โหลดเองด้วย SHA256

In [6]:
# C. Traffic: official DOH annual AADT survey CSVs (BE 2563-2568 = CE 2020-2025 where listed in the package)
T = CFG['traffic']
selected = select_aadt_resources(DOH_PKG['result']['resources'], T['survey_years_be'], T['prefer_format'])
PROVINCE_ALIASES = CFG['province_aliases']['map']  # exact audited full-string aliases -> v1 panel key (also used for geography)
meta_rows, norm_frames, TRAFFIC_ROW_AUDIT = [], [], {}
for be in T['survey_years_be']:
    res, alts = selected.get(be, (None, []))
    row = {'survey_year_be': be, 'survey_year_ce': be - 543, 'normalize_status': 'not_listed_in_package', 'rows': None,
           'duplicate_key_rows': None, 'total_mismatch_rows': None, 'catalog_created': None, 'available_from_proxy_utc': None}
    if res is None:
        GAPS.append(f'traffic: no {T["prefer_format"]} AADT resource for BE {be} in the DOH package')
        meta_rows.append(row)
        continue
    row.update(resource_meta(res))
    row.update(traffic_available_from(res, T['availability_buffer_days']))
    row['available_from_proxy'] = row['available_from_proxy_utc']  # replaces the created-only proxy of resource_meta
    row['available_from_note'] = T['availability_policy']
    row['alternative_resources'] = ';'.join(a.get('url', '') for a in alts)
    e = fetch(f'doh_aadt_{be}_csv', res['url'], 'raw/doh_aadt/' + res['url'].rsplit('/', 1)[-1], validate=nonempty_validator,
              meta={'source_group': 'traffic', 'catalog': resource_meta(res)})
    row.update(fetch_status=e.get('last_status'), sha256=e.get('sha256'), bytes=e.get('bytes'), error=e.get('error'))
    row['size_matches_catalog'] = (e.get('bytes') == res.get('size')) if e.get('bytes') and res.get('size') else None
    if not e.get('sha256'):
        row['normalize_status'] = 'unavailable'
        GAPS.append(f"traffic: BE {be} AADT download unavailable: {e.get('error')}")
        meta_rows.append(row)
        continue
    try:
        raw, enc = read_csv_bytes((EXT / e['path']).read_bytes(), T['encodings_to_try'])
    except (ValueError, pd.errors.ParserError) as ex:  # undecodable/unparseable bytes are a recorded gap
        row['normalize_status'] = f'unreadable: {type(ex).__name__}: {str(ex)[:200]}'
        GAPS.append(f"traffic: BE {be} CSV unreadable: {row['normalize_status']}")
        meta_rows.append(row)
        continue
    norm, rep = normalize_aadt(raw, T['aliases'], T['required_fields'], T['total_components'], T['heavy_components'], be)
    unit_flags = aadt_unit_flags(raw.columns, T['unsupported_unit_patterns'])
    status = rep['status']
    if 'total' in rep['mapped'] and norm_header(rep['mapped']['total']) in unit_flags:
        status = 'unsupported_total_unit'
        GAPS.append(f'traffic: BE {be} total column {rep["mapped"]["total"]!r} looks like a non-AADT unit; not normalized')
    st_rows, aud = aadt_station_rows(raw, norm, rep, PROVINCE_ALIASES, PANEL_PROVINCES)
    TRAFFIC_ROW_AUDIT[be] = aud
    st_total = st_rows.loc[st_rows['row_class'].eq('station') & st_rows['total_valid'], 'total_aadt']
    row.update(encoding=enc, rows=len(raw), n_columns=raw.shape[1], normalize_status=status, **{k: v for k, v in aud.items() if not isinstance(v, (dict, list))},
               duplicate_key_rows=aud['rows_duplicate_key'], duplicate_key_rows_including_blank_lines=rep['duplicate_key_rows'],
               total_mismatch_rows=rep['total_check'].get('n_mismatch_gt1'), heavy_share_mismatch_rows=rep['heavy_check'].get('n_mismatch_gt0_05pp'),
               negative_values=rep['negative_values'], nonnumeric_count_cells=sum(rep['nonnumeric_cells'].values()),
               nonnumeric_by_column=rep['nonnumeric_cells'], unmapped_text_headers=';'.join(rep['unmapped_text_headers']),
               unit_flagged_headers=';'.join(unit_flags), province_key_status_counts=aud['province_key_status_counts'],
               province_alias_values=aud['province_alias_values'], provinces=int(st_rows.loc[st_rows['province_key'].ne(''), 'province_key'].nunique()),
               rows_province_not_in_v1_panel=aud['province_key_status_counts'].get('not_in_v1_panel', 0) if PANEL_PROVINCES else None,
               provinces_not_in_v1_panel=';'.join(aud['provinces_not_in_v1_panel']) if PANEL_PROVINCES else None,
               station_total_aadt_median=float(st_total.median()) if len(st_total) else None)
    if status == 'normalized':
        st_rows['source_sha256'] = e['sha256']
        st_rows['available_from_proxy_utc'] = row['available_from_proxy_utc']
        norm_frames.append(st_rows)
    elif status == 'schema_incomplete':
        GAPS.append(f"traffic: BE {be} schema incomplete, missing {rep['missing_fields']} (headers kept in traffic_resource_metadata.csv)")
    if aud['rows_blank_nonrecord']:
        GAPS.append(f"traffic: BE {be} has {aud['rows_blank_nonrecord']} fully blank CSV lines: counted as non-records and excluded from station "
                    'data/statistics (raw bytes kept; not duplicate survey points)')
    for what, n in (('duplicate survey-point key rows (non-blank, complete key)', aud['rows_duplicate_key']),
                    ('incomplete-identity rows (blank route/control section/survey point/province; not merged)', aud['rows_incomplete_identity']),
                    ('invalid total values (negative/non-finite)', aud['nonblank_invalid_total_value_rows']),
                    ('invalid heavy-share values (outside 0..100/non-finite)', aud['nonblank_invalid_heavy_share_value_rows']),
                    ('rows with invalid vehicle-class counts', aud['nonblank_invalid_count_value_rows']),
                    ('non-numeric count cells', row['nonnumeric_count_cells']),
                    ('rows with province not in the v1 panel after the audited alias map', row['rows_province_not_in_v1_panel'])):
        if n:
            GAPS.append(f'traffic: BE {be} has {n} {what} (flagged and excluded from statistics, not silently fixed)')
    COVERAGE.append({'source_group': 'traffic', 'resource_id': res.get('id'), 'year': be, 'format': res.get('format'), 'rows': len(raw),
                     'n_columns': raw.shape[1], 'mapped_fields': ','.join(f'{k}={v}' for k, v in rep['mapped'].items()),
                     'missing_fields': ','.join(rep['missing_fields']), 'unmapped_headers': ','.join(rep['unmapped_text_headers']),
                     'coverage_start': f'{be - 543}-01-01', 'coverage_end': f'{be - 543}-12-31', 'status': status, 'sha256': e['sha256'],
                     'note': f"unit vehicles/day at survey points; encoding {enc}; survey year from title (BE {be}); available_from proxy "
                             f"{row['available_from_proxy_utc']} ({row['available_from_basis']})"})
    print(f"  AADT BE {be}: {status} parsed={aud['rows_parsed']} blank={aud['rows_blank_nonrecord']} station={aud['rows_station']} "
          f"dup_keys={aud['rows_duplicate_key']} incomplete={aud['rows_incomplete_identity']} valid_total={aud['station_valid_total_rows']} "
          f"keys={aud['province_key_status_counts']} total_check={rep['total_check']} available_from={row['available_from_proxy_utc']} enc={enc}", flush=True)
    meta_rows.append(row)
TRAFFIC_META = pd.DataFrame(meta_rows)
pf = CFG['preflight_files']['doh_aadt_2564_csv']
pe = (register_preflight('preflight:doh_aadt_2564_csv', pf['url'], PREFLIGHT / pf['file'], REG, 'codex_preflight_download_original_time_unknown')
      if (PREFLIGHT / pf['file']).is_file() else {})
save_reg()
fetched_2564 = TRAFFIC_META.loc[TRAFFIC_META['survey_year_be'] == 2564, 'sha256'] if 'sha256' in TRAFFIC_META else pd.Series(dtype=object)
pe_match = bool(pe) and (not fetched_2564.empty) and fetched_2564.iloc[0] == pe.get('sha256')
TRAFFIC_NORM = pd.concat(norm_frames, ignore_index=True) if norm_frames else pd.DataFrame()
TRAFFIC_PROVINCE = aadt_province_summary(TRAFFIC_NORM, PANEL_PROVINCES)
if len(TRAFFIC_NORM):
    TRAFFIC_NORM.to_csv(EXT / 'derived' / 'traffic_aadt_normalized.csv', index=False)
    DERIVED.append('derived/traffic_aadt_normalized.csv')
write_csv(REP / 'traffic_resource_metadata.csv', TRAFFIC_META)
write_csv(REP / 'traffic_province_year_summary.csv', TRAFFIC_PROVINCE)
RUN['sections']['traffic'] = {'preflight_2021_csv_matches_notebook_download': bool(pe_match), 'package_metadata_modified': DOH_PKG['result'].get('metadata_modified'),
                              'license': DOH_PKG['result'].get('license_title'), 'unit_note': T['unit_note'], 'availability_policy': T['availability_policy'],
                              'province_year_rows': len(TRAFFIC_PROVINCE), 'row_audit_by_survey_year_be': TRAFFIC_ROW_AUDIT,
                              'province_aliases': CFG['province_aliases'], 'station_schema': T['station_schema']}
if not pe_match:
    GAPS.append('traffic: Codex preflight aadt_2021.csv does not match (or could not be compared with) the notebook download of the BE2564 CSV')
GAPS.append('traffic: availability uses max(created, last_modified) + 1 day catalog proxy; it does not prove the current bytes were delivered then')
print('preflight 2021 CSV identical to notebook download:', pe_match)
TRAFFIC_META[[c for c in ('survey_year_be', 'normalize_status', 'rows_parsed', 'rows_blank_nonrecord', 'rows_station', 'rows_duplicate_key',
                          'rows_incomplete_identity', 'station_valid_total_rows', 'total_mismatch_rows', 'available_from_proxy_utc')
              if c in TRAFFIC_META]]

  AADT BE 2563: normalized parsed=2701 blank=0 station=2701 dup_keys=0 incomplete=0 valid_total=2701 keys={'v1_panel_audited_alias': 43, 'v1_panel_exact': 2658} total_check={'evaluated': True, 'n_compared': 2701, 'n_mismatch_gt1': 0} available_from=2024-06-01T07:41:10.140471+00:00 enc=utf-8-sig


  AADT BE 2564: normalized parsed=2709 blank=0 station=2709 dup_keys=0 incomplete=0 valid_total=2709 keys={'v1_panel_audited_alias': 45, 'v1_panel_exact': 2664} total_check={'evaluated': True, 'n_compared': 2709, 'n_mismatch_gt1': 0} available_from=2022-03-25T06:10:50.811162+00:00 enc=utf-8-sig


  AADT BE 2565: normalized parsed=2701 blank=0 station=2701 dup_keys=0 incomplete=0 valid_total=2701 keys={'v1_panel_audited_alias': 45, 'v1_panel_exact': 2656} total_check={'evaluated': True, 'n_compared': 2701, 'n_mismatch_gt1': 0} available_from=2024-06-01T07:48:05.556631+00:00 enc=utf-8-sig


  AADT BE 2566: normalized parsed=2710 blank=0 station=2710 dup_keys=0 incomplete=0 valid_total=2710 keys={'v1_panel_audited_alias': 44, 'v1_panel_exact': 2666} total_check={'evaluated': True, 'n_compared': 2710, 'n_mismatch_gt1': 0} available_from=2025-05-30T06:56:57.278942+00:00 enc=utf-8-sig


  AADT BE 2567: normalized parsed=4866 blank=2130 station=2736 dup_keys=0 incomplete=0 valid_total=2736 keys={'v1_panel_audited_alias': 44, 'v1_panel_exact': 2692} total_check={'evaluated': True, 'n_compared': 2736, 'n_mismatch_gt1': 0} available_from=2025-05-30T07:05:03.742919+00:00 enc=utf-8-sig


  AADT BE 2568: normalized parsed=2739 blank=0 station=2739 dup_keys=0 incomplete=0 valid_total=2739 keys={'v1_panel_audited_alias': 44, 'v1_panel_exact': 2695} total_check={'evaluated': True, 'n_compared': 2739, 'n_mismatch_gt1': 0} available_from=2026-01-21T04:28:53.557666+00:00 enc=utf-8-sig


preflight 2021 CSV identical to notebook download: True


,survey_year_be,normalize_status,rows_parsed,rows_blank_nonrecord,rows_station,rows_duplicate_key,rows_incomplete_identity,station_valid_total_rows,total_mismatch_rows,available_from_proxy_utc
0,2563,normalized,2701,0,2701,0,0,2701,0,2024-06-01T07:41:10.140471+00:00
1,2564,normalized,2709,0,2709,0,0,2709,0,2022-03-25T06:10:50.811162+00:00
2,2565,normalized,2701,0,2701,0,0,2701,0,2024-06-01T07:48:05.556631+00:00
3,2566,normalized,2710,0,2710,0,0,2710,0,2025-05-30T06:56:57.278942+00:00
4,2567,normalized,4866,2130,2736,0,0,2736,0,2025-05-30T07:05:03.742919+00:00
5,2568,normalized,2739,0,2739,0,0,2739,0,2026-01-21T04:28:53.557666+00:00


## D. ภูมิศาสตร์และจุดสอบถามอากาศรายจังหวัด

- ใช้สำเนาที่ **pin ด้วย sha256** ใน config ก่อน (`geography.pinned_sources`): สำเนา preflight ของ Codex ที่ลงทะเบียนแล้ว (บันทึก cache mtime / verified_at; เวลาดาวน์โหลดเดิมไม่ทราบ) หรือ cache ของ notebook ที่ hash ตรง pin — **ไม่ต้องใช้เครือข่าย** และทำงาน offline ได้; ดาวน์โหลด HTTPS เฉพาะเมื่อไม่มีสำเนาเลย (สภาพแวดล้อมใหม่) หรือสั่ง refresh
- ไบต์ที่ไม่ตรง pin ถูกเก็บเป็น version ใน registry แต่ **ไม่ถูกใช้และไม่ถูกเปลี่ยนป้าย**; แหล่งที่จำเป็น (GeoJSON / ตารางจังหวัด) ไม่ตรง pin จะหยุดการรันเพื่อให้ตรวจทาน
- ขอบเขต ADM1 จาก geoBoundaries (GeoJSON ที่ pin commit) — มาจาก OSM/ODbL (ชุมชน) **ไม่ใช่** ข้อมูลทางการของรัฐไทย; metadata ของ API อ่านจากสำเนาที่ pin
- ตารางรหัส/ชื่อจังหวัดไทย–อังกฤษจาก thailand-geography-json; ข้อมูล repo/commit จาก GitHub API เป็นข้อมูลเสริม (timeout สั้น ไม่ retry) ถ้าล้มเหลวบันทึกเป็นช่องว่างและไม่ขวางการสร้างจุด
- จับคู่ shape → จังหวัดด้วย ISO `TH-xx` = รหัสจังหวัด ก่อน ถ้าไม่มีจึงใช้ชื่ออังกฤษที่ normalize แล้ว **ตรงตัว** เท่านั้น ไม่เดาแบบ fuzzy; ที่จับคู่ไม่ได้แสดงชัดเจน
- จุด = `representative_point()` ของ shapely ซึ่งรับประกันว่าอยู่ **ภายใน** รูปจังหวัด (ตรวจ `contains` ซ้ำ) เป็นจุด proxy สำหรับสอบถามอากาศ ไม่ใช่ตำแหน่งถนน/เหตุการณ์ และไม่ใช้พิกัดจากอุบัติเหตุ
- เทียบกับ 77 จังหวัดใน panel v1 (อ่านอย่างเดียว) ผ่าน `province_key` ซึ่งใช้แผนที่ alias ที่ตรวจแล้วแบบตรงตัวชุดเดียวกับข้อมูลจราจร (`province_aliases`); `name_th` คงข้อความเดิมจากตาราง

In [7]:
# D. Geography: geoBoundaries ADM1 (pinned release) + maintained province code table -> representative query points
# Local first: sha256-pinned Codex preflight copies (or a pin-verified notebook cache) are used without network; HTTPS download only
# when neither exists (fresh environment) or with refresh. Optional live metadata calls use a short timeout and never block the points.
G = CFG['geography']
PINS = G['pinned_sources']
POINTS = pd.DataFrame(columns=POINT_COLUMNS + ['province_key', 'province_key_status'])
GEO_STATUS = {'summary': 'geography not collected'}
GEO_META_NET = dict(NET, **G['metadata_http'])
GEO_DOWNLOAD_NET = dict(NET, **G['download_http'])


def fetch_net(net):
    def _f(res_id, url, rel, **kw):
        e = try_fetch(res_id, url, rel, EXT, REG, net, BUDGET, refresh=REFRESH, offline=OFFLINE, **kw)
        save_reg()
        return e
    return _f


GEO_SOURCES = {}
geo_used = {}
for key, net, kw in (('geoboundaries_api', GEO_META_NET, {'validate': json_validator()}),
                     ('geoboundaries_tha_adm1_geojson', GEO_DOWNLOAD_NET, {'validate': json_validator(), 'max_bytes': G['max_geojson_bytes']}),
                     ('thailand_provinces_json', GEO_DOWNLOAD_NET, {'validate': json_validator()})):
    e, path, info = pinned_source(key, PINS[key], PREFLIGHT, EXT, REG, fetch_net(net), refresh=REFRESH, **kw)
    save_reg()
    GEO_SOURCES[key] = info
    geo_used[key] = (e, path)
    if info['error']:
        GAPS.append(f"geography: {info['error']}")
    if info['remote_matches_pin'] is False:
        GAPS.append(f"geography: {key} remote bytes {info['remote_sha256'][:12]} differ from the pin {info['expected_sha256'][:12]}; kept as a "
                    'registry version, NOT used; review before changing the pin')
    print(f"  {key}: used={info['used']} sha256={(info['used_sha256'] or '-')[:12]} remote_checked={info['remote_checked']} "
          f"remote_matches_pin={info['remote_matches_pin']}", flush=True)

api_meta = {}
api_e, api_path = geo_used['geoboundaries_api']
if api_path is not None:
    a = json.loads(Path(api_path).read_text('utf-8'))
    a = a[0] if isinstance(a, list) and a else a
    api_meta = {k: a.get(k) for k in ('boundaryID', 'boundaryISO', 'boundaryType', 'boundaryYearRepresented', 'buildDate', 'sourceDataUpdateDate',
                                      'boundarySource', 'boundaryLicense', 'licenseDetail', 'licenseSource', 'admUnitCount', 'gjDownloadURL')}
    api_meta['metadata_source'] = GEO_SOURCES['geoboundaries_api']['used']
api_meta['pinned_gj_download_url'] = G['pinned_gj_download_url']
api_meta['current_api_url_matches_pinned'] = (api_meta.get('gjDownloadURL') == G['pinned_gj_download_url']) if api_meta.get('gjDownloadURL') else None

# optional live context about the maintained province table: short timeout, no retry; failures are recorded gaps, never blockers
meta_fetch = fetch_net(GEO_META_NET)
repo_e = meta_fetch('thailand_geography_repo_api', G['province_table_repo_api'], 'raw/geo/thailand_geography_repo.json', validate=json_validator())
commit_e = meta_fetch('thailand_geography_provinces_commit_api', G['province_table_commit_api'], 'raw/geo/thailand_geography_provinces_commit.json',
                      validate=json_validator())
table_meta = {}
for what, e in (('repo', repo_e), ('commit', commit_e)):
    p = entry_file(e, EXT)
    table_meta[f'{what}_api_status'] = e.get('last_status')
    table_meta[f'{what}_api_retrieved_at_utc'] = e.get('retrieved_at_utc')
    if p is None:
        GAPS.append(f"geography: optional {what} metadata for the province table unavailable ({str(e.get('error'))[:160]}); points unaffected")
        continue
    o = json.loads(p.read_text('utf-8'))
    if what == 'repo':
        table_meta['license'] = (o.get('license') or {}).get('spdx_id')
        table_meta['default_branch'] = o.get('default_branch')
    elif isinstance(o, list) and o:
        table_meta['latest_commit_touching_file_at_fetch'] = o[0].get('sha')
        table_meta['latest_commit_date'] = ((o[0].get('commit') or {}).get('committer') or {}).get('date')
table_meta['revision_note'] = ('bytes are identified by the configured sha256 pin; the raw URL follows the main branch and the commit API answer '
                               'is from its own fetch time, so it does not prove which revision the pinned bytes came from')

(gj_e, gj_path), (pv_e, pv_path) = geo_used['geoboundaries_tha_adm1_geojson'], geo_used['thailand_provinces_json']
if gj_path is not None and pv_path is not None:
    gj = json.loads(Path(gj_path).read_text('utf-8'))
    shapes = representative_points(gj['features'])
    prov, keys = province_table(json.loads(Path(pv_path).read_text('utf-8')))
    j = join_shapes(shapes, prov)
    pts_all = shapes.merge(j, on='shape_index').merge(prov, on='province_code', how='left')
    pk = province_keys(pts_all['name_th'], PROVINCE_ALIASES, PANEL_PROVINCES)  # name_th stays the original table text
    pts_all['province_key'] = pk['province_key'].where(pk['province_key_status'].ne('blank'), None).values
    pts_all['province_key_status'] = pk['province_key_status'].values
    pts_all['in_v1_panel'] = pts_all['province_key_status'].isin(['v1_panel_exact', 'v1_panel_audited_alias'])
    pts_all['geometry_source'] = 'geoBoundaries gbOpen THA ADM1 @ ' + G['pinned_gj_download_url']
    pts_all['geometry_sha256'] = gj_e['sha256']
    pts_all['province_table_sha256'] = pv_e['sha256']
    pts_all['point_method'] = 'shapely_representative_point'
    pts_all['point_meaning'] = 'province label proxy query point; not a road/event location'
    POINTS = (pts_all[POINT_COLUMNS + ['province_key', 'province_key_status']]
              .sort_values(['province_code', 'shape_index'], na_position='last').reset_index(drop=True))
    unresolved = POINTS[POINTS['province_code'].isna()]
    missing_panel = sorted(set(PANEL_PROVINCES) - set(POINTS['province_key'].dropna()))
    extra = sorted(set(POINTS.loc[POINTS['province_key_status'].eq('not_in_v1_panel'), 'name_th'].dropna())) if PANEL_PROVINCES else []
    aliased = POINTS.loc[POINTS['province_key_status'].eq('v1_panel_audited_alias'), ['name_th', 'province_key']].drop_duplicates()
    GEO_STATUS = {'summary': (f"shapes {len(shapes)} (expected {G['expected_provinces']}); joined by ISO code "
                              f"{int((POINTS['join_method'] == 'iso_3166_2_code').sum())}, by exact normalized English name "
                              f"{int((POINTS['join_method'] == 'exact_normalized_english_name').sum())}, unresolved {len(unresolved)}; "
                              f"points inside polygon {int(POINTS['point_within_polygon'].astype(bool).sum())}; "
                              f"v1 panel provinces without a point: {missing_panel or 'none'}; point provinces not in panel: {extra or 'none'}; "
                              f"audited aliases used: {dict(zip(aliased['name_th'], aliased['province_key'])) or 'none'}"),
                  'api_metadata': api_meta, 'province_table': {**keys, **table_meta}, 'unresolved_shapes': unresolved['shape_name'].tolist(),
                  'panel_provinces_missing': missing_panel, 'points_not_in_panel': extra, 'pinned_sources': GEO_SOURCES,
                  'quality_label': G['source_quality_label']}
    if len(unresolved) or missing_panel or extra or len(shapes) != G['expected_provinces']:
        GAPS.append('geography join incomplete: ' + GEO_STATUS['summary'])
    COVERAGE.append({'source_group': 'geography', 'resource_id': 'geoboundaries_tha_adm1_geojson_pinned', 'year': api_meta.get('boundaryYearRepresented'),
                     'format': 'geojson', 'rows': len(shapes), 'n_columns': None, 'mapped_fields': 'shapeName,shapeISO,shapeID,geometry',
                     'missing_fields': None, 'unmapped_headers': None, 'coverage_start': None, 'coverage_end': None,
                     'status': GEO_SOURCES['geoboundaries_tha_adm1_geojson']['used'], 'sha256': gj_e['sha256'], 'note': G['source_quality_label']})
    COVERAGE.append({'source_group': 'geography', 'resource_id': 'thailand_geography_provinces_json', 'year': None, 'format': 'json',
                     'rows': len(prov), 'n_columns': 3, 'mapped_fields': ','.join(f'{k}={v}' for k, v in keys.items()), 'missing_fields': None,
                     'unmapped_headers': None, 'coverage_start': None, 'coverage_end': None,
                     'status': GEO_SOURCES['thailand_provinces_json']['used'], 'sha256': pv_e['sha256'],
                     'note': f"license {table_meta.get('license')}; {table_meta['revision_note']}"})
else:
    GAPS.append('geography unavailable (pinned geojson or province table has no verified local copy and could not be downloaded): '
                'weather collection is blocked; this is a source/network gap, not evidence that the archive is missing')
    GEO_STATUS.update(api_metadata=api_meta, pinned_sources=GEO_SOURCES)
write_csv(REP / 'province_points.csv', POINTS)
RUN['sections']['geography'] = GEO_STATUS
print(GEO_STATUS['summary'])
POINTS.head()

  geoboundaries_api: used=codex_preflight_registered_pin_verified sha256=10119cda1b99 remote_checked=False remote_matches_pin=None


  geoboundaries_tha_adm1_geojson: used=codex_preflight_registered_pin_verified sha256=145beb11e527 remote_checked=False remote_matches_pin=None


  thailand_provinces_json: used=codex_preflight_registered_pin_verified sha256=ec29779c963d remote_checked=False remote_matches_pin=None


shapes 77 (expected 77); joined by ISO code 77, by exact normalized English name 0, unresolved 0; points inside polygon 77; v1 panel provinces without a point: none; point provinces not in panel: none; audited aliases used: {'พระนครศรีอยุธยา': 'อยุธยา'}


,province_code,name_th,name_en,shape_name,shape_iso,shape_id,shape_index,join_method,lat,lon,point_within_polygon,geometry_valid,in_v1_panel,point_method,point_meaning,geometry_source,geometry_sha256,province_table_sha256,province_key,province_key_status
0,10,กรุงเทพมหานคร,Bangkok,Bangkok,TH-10,36821470B28205534186964,14,iso_3166_2_code,13.724433,100.603385,True,True,True,shapely_representative_point,province label proxy query point; not a road/e...,geoBoundaries gbOpen THA ADM1 @ https://github...,145beb11e52785a42e16997c92a65426b3df8d009941db...,ec29779c963de89a29f72a37eabcd39f89d1c6837586b5...,กรุงเทพมหานคร,v1_panel_exact
1,11,สมุทรปราการ,Samut Prakan,Samut Prakan Province,TH-11,36821470B35369831436030,69,iso_3166_2_code,13.598574,100.719904,True,True,True,shapely_representative_point,province label proxy query point; not a road/e...,geoBoundaries gbOpen THA ADM1 @ https://github...,145beb11e52785a42e16997c92a65426b3df8d009941db...,ec29779c963de89a29f72a37eabcd39f89d1c6837586b5...,สมุทรปราการ,v1_panel_exact
2,12,นนทบุรี,Nonthaburi,Nonthaburi Province,TH-12,36821470B99224408335371,38,iso_3166_2_code,13.964864,100.383655,True,True,True,shapely_representative_point,province label proxy query point; not a road/e...,geoBoundaries gbOpen THA ADM1 @ https://github...,145beb11e52785a42e16997c92a65426b3df8d009941db...,ec29779c963de89a29f72a37eabcd39f89d1c6837586b5...,นนทบุรี,v1_panel_exact
3,13,ปทุมธานี,Pathum Thani,Pathum Thani Province,TH-13,36821470B26201996409674,58,iso_3166_2_code,14.095273,100.627655,True,True,True,shapely_representative_point,province label proxy query point; not a road/e...,geoBoundaries gbOpen THA ADM1 @ https://github...,145beb11e52785a42e16997c92a65426b3df8d009941db...,ec29779c963de89a29f72a37eabcd39f89d1c6837586b5...,ปทุมธานี,v1_panel_exact
4,14,พระนครศรีอยุธยา,Phra Nakhon Si Ayutthaya,Phra Nakhon Si Ayutthaya Province,TH-14,36821470B14521381139968,53,iso_3166_2_code,14.397221,100.531945,True,True,True,shapely_representative_point,province label proxy query point; not a road/e...,geoBoundaries gbOpen THA ADM1 @ https://github...,145beb11e52785a42e16997c92a65426b3df8d009941db...,ec29779c963de89a29f72a37eabcd39f89d1c6837586b5...,อยุธยา,v1_panel_audited_alias


## E. พยากรณ์อากาศย้อนหลังที่เก็บถาวร (Open-Meteo Previous Runs API, JMA GSM)

- ขอเฉพาะ 2 ตัวแปร `temperature_2m_previous_day2`, `precipitation_previous_day2` ล็อกโมเดล `jma_gsm` (ไม่ใช้ best-match ที่เปลี่ยนโมเดลอัตโนมัติ) เขตเวลา Asia/Bangkok ปี 2021–2025 สำหรับจุดตัวแทนของ 77 จังหวัด แบ่ง chunk = 1 ปี × 5 จุด
- timeout 120 วินาที, retry สูงสุด 3 ครั้ง, เว้นระยะระหว่างคำขอ, จำกัดจำนวนคำขอต่อรอบ; โดน HTTP 429 (ไม่มี Retry-After สั้น) จะ **หยุด** และบันทึกความครอบคลุมบางส่วน รันซ้ำเพื่อเก็บต่อ (chunk ที่เสร็จแล้วใช้ซ้ำเมื่อ SHA256 ตรง)
- ไม่ใช้ ERA5/ค่าสังเกตการณ์ของวันเป้าหมายแทน; ค่าว่างคงเป็นค่าว่าง
- `nominal_issue_bound_utc = valid_time_utc − 48 ชม.`; สมมติ buffer เผยแพร่ 6 ชม.; แถวรวมใช้ได้เมื่อ `latest_issue_bound + 6 ชม. ≤ (วันเป้าหมาย − 1) 18:00 น.` — ระยะ lead ของ API ไม่ได้พิสูจน์ความหน่วงการส่งข้อมูลจริงในอดีต
- รวมรายวัน และราย 6 ชม. (00–06/06–12/12–18/18–24): `rain_sum`, `temp_mean` คำนวณเฉพาะเมื่อครบทุกชั่วโมง (24 หรือ 6) มิฉะนั้นเป็นค่าว่างพร้อมธง missing; บันทึกพิกัดที่ขอเทียบกับพิกัด grid ที่ได้

เซลล์นี้อาจใช้เวลาหลายนาทีและพิมพ์ความคืบหน้าทุก chunk

In [8]:
# E. Archived forecast weather: Open-Meteo Previous Runs API, model locked to jma_gsm, lead previous_day2 (~48 h)
W = CFG['weather']
NET_W = dict(NET, timeout_seconds=W['timeout_seconds'], max_retries=W['max_retries'], max_response_bytes=W['max_response_bytes'])
WBUDGET = RequestBudget(W['max_requests_per_run'], parent=BUDGET)
_probe = CFG['preflight_files']['weather_probe']
if (PREFLIGHT / _probe['file']).is_file():
    register_preflight('preflight:weather_jma_2021_probe', _probe['url'], PREFLIGHT / _probe['file'], REG, 'codex_preflight_probe_original_time_unknown')
save_reg()
pts = POINTS[POINTS['province_code'].notna() & POINTS['point_within_polygon'].astype(bool)].sort_values('province_code').reset_index(drop=True)
batches = [pts.iloc[i:i + int(W['locations_per_request'])] for i in range(0, len(pts), int(W['locations_per_request']))]
plan = [(y, bi, b) for y in W['years'] for bi, b in enumerate(batches)]
print(f'weather plan: {len(pts)} province points x {len(W["years"])} years -> {len(plan)} chunks; '
      f'approx. Open-Meteo call weight per chunk {estimate_call_weight(int(W["locations_per_request"]), 366, len(W["hourly_variables"]))} (estimate); '
      f'cached chunks are reused without waiting, network chunks wait {W["min_seconds_between_requests"]} s', flush=True)
NEG_POLICY = W['negative_value_policy']  # 'mask': a negative precipitation hour -> NaN in that hour only; structural checks stay strict
frames, chunk_log, stop_reason, last_req, fails, t_start, weight_used = [], [], None, 0.0, 0, time.time(), 0.0
q_flags, q_audit = [], []
for y, bi, b in plan:
    rid = f'weather_jma_gsm_{y}_b{bi:02d}'
    rel = f'raw/weather/jma_gsm_{y}_b{bi:02d}.json'
    url = weather_url(W['base_url'], b, f'{y}-01-01', f'{y}-12-31', W)
    e0 = REG['resources'].get(rid) or {}
    cached = bool(e0.get('sha256') and e0.get('path') and (EXT / e0['path']).is_file() and e0.get('url') == url)
    network = (not cached or REFRESH) and not OFFLINE
    rec = {'chunk': rid, 'year': y, 'batch': bi, 'province_codes': ','.join(str(int(c)) for c in b['province_code']), 'url': url}
    wt = estimate_call_weight(len(b), (dt.date(y, 12, 31) - dt.date(y, 1, 1)).days + 1, len(W['hourly_variables']))
    if network and not stop_reason and weight_used + wt > float(W['max_weighted_calls_per_run']):
        stop_reason = 'quota_budget_per_run_reached_resume_later'
    if stop_reason and network:
        rec['status'] = 'not_attempted_' + stop_reason
        chunk_log.append(rec)
        continue
    if e0.get('sha256') and e0.get('url') != url:
        rec.update(status='failed', error='registry chunk URL differs from the planned URL (points changed?); move the old chunk aside to recollect')
        chunk_log.append(rec)
        print(f"  [{len(chunk_log)}/{len(plan)}] {rid}: {rec['error']}", flush=True)
        continue
    if network:
        wait = float(W['min_seconds_between_requests']) - (time.time() - last_req)
        if wait > 0:
            time.sleep(wait)
        weight_used += wt
    try:
        e = cached_fetch(rid, url, rel, EXT, REG, NET_W, WBUDGET, refresh=REFRESH, offline=OFFLINE,
                         validate=weather_validator(len(b)), meta={'source_group': 'weather', 'model': W['model']})
        if network:
            last_req = time.time()
        payload = json.loads((EXT / e['path']).read_text('utf-8'))  # hash-verified cached bytes; sanitized in memory only, never rewritten
        validate_weather_payload(payload, b, W['hourly_variables'], W['hourly_units'], f'{y}-01-01', f'{y}-12-31', [W['rain_var']],
                                 float(W['max_grid_distance_deg']), negative_policy=NEG_POLICY)
        f = normalize_weather_payload(payload, b, W['hourly_variables'], W['model'], W['timezone'], CFG['utc_offset_seconds'], W['lead_hours'])
        f, fl, qa = apply_weather_value_policy(f, W['hourly_variables'], [W['rain_var']])
        frames.append(f)
        q_flags.append(fl.assign(chunk=rid, year=y))
        q_audit.append(qa.assign(chunk=rid, year=y, chunk_status=e['last_status']))
        fails = 0  # value-quality masking is not a failure; only structural/network errors below count towards repeated_failures
        rqa = qa[qa['variable'] == W['rain_var']]
        rec.update(status=e['last_status'], sha256=e['sha256'], rows=len(f),
                   null_temp=int(f[W['temp_var']].isna().sum()), null_rain=int(f[W['rain_var']].isna().sum()),
                   rain_negative_masked=int(rqa['n_negative_masked'].sum()), rain_null_in_source=int(rqa['n_null_in_source'].sum()),
                   rain_min_raw=float(rqa['min_raw'].min()) if rqa['min_raw'].notna().any() else None)
    except (RateLimited, BudgetExhausted) as ex:
        last_req = time.time()
        stop_reason = 'rate_limited' if isinstance(ex, RateLimited) else 'request_budget_exhausted'
        rec.update(status=stop_reason, error=str(ex)[:500])
    except (FetchError, ValueError, KeyError) as ex:
        last_req = time.time()
        fails += 1
        rec.update(status='failed', error=f'{type(ex).__name__}: {str(ex)[:500]}')
        if fails >= int(W['consecutive_failures_to_stop']):
            stop_reason = 'repeated_failures'
    save_reg()
    chunk_log.append(rec)
    print(f"  [{len(chunk_log)}/{len(plan)}] {y} batch {bi:02d} ({len(b)} provinces): {rec['status']} rows={rec.get('rows', 0)} "
          f"null_temp={rec.get('null_temp', '-')} null_rain={rec.get('null_rain', '-')} rain_negative_masked={rec.get('rain_negative_masked', '-')} "
          f"requests={WBUDGET.used} "
          f"elapsed={time.time() - t_start:.0f}s", flush=True)

W_HOURLY = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
W_DAILY, W_6H = pd.DataFrame(), pd.DataFrame()
cov = []
if len(W_HOURLY):
    hourly_rel = write_table(EXT / 'derived' / 'weather_hourly_jma_gsm', W_HOURLY)
    DERIVED.append(Path(hourly_rel).relative_to(EXT).as_posix())
    agg_args = dict(temp_col=W['temp_var'], rain_col=W['rain_var'], buffer_hours=W['assumed_publication_buffer_hours'],
                    issue_hour_local=CFG['forecast_issue']['issue_hour_local'], utc_offset_seconds=CFG['utc_offset_seconds'])
    W_DAILY = aggregate_weather(W_HOURLY, 24, **agg_args)
    W_6H = aggregate_weather(W_HOURLY, 6, **agg_args)
    for name, df in (('weather_daily_jma_gsm.csv.gz', W_DAILY), ('weather_6h_jma_gsm.csv.gz', W_6H)):
        write_csv_gz(EXT / 'derived' / name, df)  # fixed gzip header: reruns with identical data give identical bytes
        DERIVED.append('derived/' + name)
# value-quality lineage: hour-level masked hours (raw value kept) under the ignored external root; tracked per chunk/province audit in reports
W_QFLAGS = pd.concat(q_flags, ignore_index=True) if q_flags else pd.DataFrame(columns=['province_code', 'valid_time_local', 'variable', 'raw_value',
                                                                                         'quality_flag', 'chunk', 'year'])
if len(W_HOURLY):
    write_csv_gz(EXT / 'derived' / 'weather_value_quality_flags_jma_gsm.csv.gz', W_QFLAGS)
    DERIVED.append('derived/weather_value_quality_flags_jma_gsm.csv.gz')
qa_rows = [pd.concat(q_audit, ignore_index=True)] if q_audit else []
for c in chunk_log:  # chunks without accepted data: availability/network/structural reason, never a value-quality reason
    if not c.get('sha256'):
        qa_rows.append(pd.DataFrame([{'province_code': int(p), 'variable': v, 'n_returned_hours': 0, 'n_valid': 0, 'n_null_in_source': 0,
                                      'n_negative_masked': 0, 'min_raw': None, 'max_raw': None, 'quality_reason': 'no_accepted_data:' + c['status'],
                                      'chunk': c['chunk'], 'year': c['year'], 'chunk_status': c['status']}
                                     for p in c['province_codes'].split(',') for v in W['hourly_variables']]))
W_QUALITY = pd.concat(qa_rows, ignore_index=True) if qa_rows else pd.DataFrame(columns=WEATHER_QUALITY_COLUMNS + ['chunk', 'year', 'chunk_status'])
W_QUALITY = W_QUALITY[['year', 'chunk', 'chunk_status'] + WEATHER_QUALITY_COLUMNS].sort_values(['year', 'province_code', 'variable'], ignore_index=True)
write_csv(REP / 'weather_value_quality.csv', W_QUALITY)
by_key = {k: g for k, g in W_HOURLY.groupby(['province_code', W_HOURLY['valid_time_local'].dt.year])} if len(W_HOURLY) else {}
chunk_of = {(c['year'], int(p)): c for c in chunk_log for p in c['province_codes'].split(',')}
for y in W['years']:
    exp_hours = len(pd.date_range(f'{y}-01-01 00:00', f'{y}-12-31 23:00', freq='h'))
    for r in pts.itertuples():
        sub = by_key.get((int(r.province_code), y))
        n = 0 if sub is None else len(sub)
        nt = int(sub[W['temp_var']].notna().sum()) if n else 0
        nr = int(sub[W['rain_var']].notna().sum()) if n else 0
        nm = int(sub[W['rain_var'] + '_quality'].eq('negative_value_masked').sum()) if n else 0
        ch = chunk_of.get((y, int(r.province_code)), {})
        if n == exp_hours and nt == exp_hours and nr == exp_hours:
            st, why = 'complete', ''
        elif n:
            st = 'partial_value_quality_masked' if nm else 'partial_or_null'
            null_any = (n - nr - nm) > 0 or (n - nt) > 0
            why = ';'.join(s for s, c in (('rain_negative_value_masked', nm), ('null_in_source', null_any), ('hours_not_returned', n != exp_hours)) if c)
        else:
            st, why = 'not_collected', 'no_accepted_data:' + str(ch.get('status', 'not_planned'))
        cov.append({'year': y, 'province_code': int(r.province_code), 'name_en': r.name_en, 'requested_lat': r.lat, 'requested_lon': r.lon,
                    'grid_lat': sub['grid_lat'].iloc[0] if n else None, 'grid_lon': sub['grid_lon'].iloc[0] if n else None,
                    'n_expected_hours': exp_hours, 'n_returned_hours': n, 'n_valid_temp': nt, 'n_valid_rain': nr,
                    'n_rain_negative_masked': nm, 'chunk': ch.get('chunk'), 'status': st, 'missing_reason': why})
W_COVERAGE = pd.DataFrame(cov)
write_csv(REP / 'weather_coverage.csv', W_COVERAGE)
write_csv(REP / 'weather_chunks.csv', pd.DataFrame(chunk_log))
n_ok = sum(1 for c in chunk_log if c.get('sha256'))
n_complete = int((W_COVERAGE['status'] == 'complete').sum()) if len(W_COVERAGE) else 0
n_retained = int((W_COVERAGE['n_returned_hours'] > 0).sum()) if len(W_COVERAGE) else 0
n_masked = int((W_QFLAGS['quality_flag'] == 'negative_value_masked').sum())
W_STATUS = {
    'readiness_status': 'blocked' if n_ok == 0 else ('collected' if n_complete == len(W_COVERAGE) else 'partial'),
    'requests_used': WBUDGET.used, 'estimated_weighted_calls_this_run': round(weight_used, 1), 'stop_reason': stop_reason,
    'quota_limited': stop_reason in ('quota_budget_per_run_reached_resume_later', 'rate_limited'), 'chunks_ok': n_ok, 'chunks_total': len(plan),
    'negative_value_policy': NEG_POLICY, 'rain_negative_hours_masked': n_masked, 'province_years_retained': n_retained,
    'province_years_with_masked_hours': int((W_COVERAGE['n_rain_negative_masked'] > 0).sum()) if len(W_COVERAGE) else 0,
    'summary': (f'JMA GSM previous_day2 chunks with data: {n_ok}/{len(plan)}; province-years retained: {n_retained}/{len(W_COVERAGE)}, '
                f'complete (all hours valid): {n_complete}/{len(W_COVERAGE)}; negative precipitation hours masked to NaN (raw JSON kept): '
                f'{n_masked}; stop reason: {stop_reason or "none"}; '
                f'daily rows {len(W_DAILY)} (complete rain {int(W_DAILY["rain_sum"].notna().sum()) if len(W_DAILY) else 0}); '
                f'issue bound = valid - {W["lead_hours"]} h, buffer {W["assumed_publication_buffer_hours"]} h, all issue_bound_ok='
                f'{bool(W_DAILY["issue_bound_ok"].all()) if len(W_DAILY) else None}'),
}
if len(W_DAILY):
    s = W_DAILY.iloc[[0, len(W_DAILY) // 2, len(W_DAILY) - 1]][['province_code', 'date', 'latest_issue_bound_utc', 'issue_time_utc', 'issue_bound_ok']]
    W_STATUS['sample_lead_bounds'] = s.to_dict('records')
if n_complete < len(W_COVERAGE):
    GAPS.append(f'weather coverage incomplete: {n_complete}/{len(W_COVERAGE)} province-years complete, {n_retained} retained '
                f'(stop reason: {stop_reason or "none"}; value-quality masked hours: {n_masked}); see weather_coverage.csv missing_reason; rerun to resume')
if n_masked:
    GAPS.append(f'weather value quality: {n_masked} negative precipitation hours (raw min {W_QFLAGS["raw_value"].min()} mm) masked to NaN in that hour only; '
                'raw JSON unchanged; any 6 h/day rain window containing one is NaN (rain_missing_reason); temperature kept')
GAPS.append('weather: Open-Meteo previous_day2 lead is nominal; historical public delivery latency is not verified (6 h buffer assumed)')
COVERAGE.append({'source_group': 'weather', 'resource_id': 'open_meteo_previous_runs_jma_gsm', 'year': ','.join(map(str, W['years'])),
                 'format': 'json', 'rows': len(W_HOURLY), 'n_columns': len(W_HOURLY.columns), 'mapped_fields': ','.join(W['hourly_variables']),
                 'missing_fields': None, 'unmapped_headers': None,
                 'coverage_start': str(W_HOURLY['valid_time_local'].min()) if len(W_HOURLY) else None,
                 'coverage_end': str(W_HOURLY['valid_time_local'].max()) if len(W_HOURLY) else None,
                 'status': W_STATUS['readiness_status'], 'sha256': None, 'note': f'model locked to {W["model"]}; tz {W["timezone"]}'})
RUN['sections']['weather'] = dict(W_STATUS)
print(W_STATUS['summary'])

weather plan: 77 province points x 5 years -> 80 chunks; approx. Open-Meteo call weight per chunk 130.7 (estimate); cached chunks are reused without waiting, network chunks wait 20 s


  [1/80] 2021 batch 00 (5 provinces): cache_hit_hash_verified rows=43800 null_temp=0 null_rain=0 rain_negative_masked=0 requests=0 elapsed=0s


  [2/80] 2021 batch 01 (5 provinces): cache_hit_hash_verified rows=43800 null_temp=0 null_rain=0 rain_negative_masked=0 requests=0 elapsed=0s


  [3/80] 2021 batch 02 (5 provinces): cache_hit_hash_verified rows=43800 null_temp=0 null_rain=6 rain_negative_masked=6 requests=0 elapsed=0s


  [4/80] 2021 batch 03 (5 provinces): cache_hit_hash_verified rows=43800 null_temp=0 null_rain=0 rain_negative_masked=0 requests=0 elapsed=0s


  [5/80] 2021 batch 04 (5 provinces): cache_hit_hash_verified rows=43800 null_temp=0 null_rain=6 rain_negative_masked=6 requests=0 elapsed=0s

... (35 outputs left out)

  [41/80] 2023 batch 08 (5 provinces): downloaded rows=43800 null_temp=0 null_rain=18 rain_negative_masked=18 requests=34 elapsed=895s


  [42/80] 2023 batch 09 (5 provinces): downloaded rows=43800 null_temp=0 null_rain=12 rain_negative_masked=12 requests=35 elapsed=920s


/var/folders/0j/7j3x0lgn1d7gnzl5z6xjmvdr0000gn/T/ipykernel_97429/1678357534.py:105: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  W_QUALITY = pd.concat(qa_rows, ignore_index=True) if qa_rows else pd.DataFrame(columns=WEATHER_QUALITY_COLUMNS + ['chunk', 'year', 'chunk_status'])


JMA GSM previous_day2 chunks with data: 42/80; province-years retained: 204/385, complete (all hours valid): 172/385; negative precipitation hours masked to NaN (raw JSON kept): 222; stop reason: quota_budget_per_run_reached_resume_later; daily rows 74460 (complete rain 74335); issue bound = valid - 48 h, buffer 6 h, all issue_bound_ok=True


## F. ปฏิทินวันหยุดธนาคาร (ธปท.) จากการตรวจด้วยตา + เทศกาลวันที่คงที่

- PDF ของ ธปท. ปี 2022–2025 และประกาศแก้ไขปลายปี 2023 เป็น **ภาพล้วน** (pypdf ได้ข้อความว่าง) ข้อความว่าง **ไม่ได้แปลว่าไม่มีวันหยุด** จึงใช้รายการที่ Codex ตรวจด้วยตาและถอดไว้ใน `config/external_sources_v2.json` (`calendar.sources[].entries`) พร้อมหน้าที่อ้างอิง วันที่หนังสือเวียน และผู้ตรวจ (ดู `workflow/calendar-source-verification.md`)
- ปี 2021 ใช้ภาพ infographic ทางการที่ฝังในหน้าข่าว ธปท. 6 ม.ค. 2021 (อนุญาตเฉพาะ URL asset นี้บนโฮสต์ scene7 พร้อมหน้าแม่ใน www.bot.or.th) `known_from` = 2021-01-06 เท่านั้น (ไม่ย้อนไปปี 2020) เป็น proxy สำหรับงานวิจัย
- รายการจะ **ใช้ได้** เมื่อไฟล์ที่ตรวจด้วยตา (สำเนา preflight) มี SHA256 ตรงกับไฟล์ที่ดาวน์โหลด/แคช (ถ้าไม่ตรง = ไม่ใช้ และบันทึกเป็นช่องว่าง) สำหรับหน้า HTML ต้องมี snapshot และพบวันที่ทุกรายการในข้อความ
- ระเบียนเป็นการกระทำ `add`/`remove` ที่มี `known_from` (ใช้ได้ตั้งแต่ **สิ้นวัน** นั้นตามเวลาท้องถิ่น) มุมมอง as-of เล่นซ้ำตามลำดับ ไม่ใช่ union ตลอดกาล เช่น 2024-01-02 ถูกลบและ 2023-12-29 ถูกเพิ่มตั้งแต่ 2023-11-14 00:00 น.
- ขอบเขตคือ **วันหยุดธนาคารระดับประเทศ ไม่ใช่วันหยุดราชการ** (`is_national_public_holiday` ว่าง); ปีที่ยังไม่รู้ปฏิทิน = ไม่ทราบ ไม่ใช่วันทำงานปกติ; ไม่อ้างว่าครบถ้วน
- เทศกาลสงกรานต์ 13–15 เม.ย. และ 31 ธ.ค./1 ม.ค. เป็นกฎปฏิทินที่รู้ล่วงหน้า ใช้ได้โดยไม่ต้องเป็นวันหยุดธนาคาร

In [9]:
# F. Calendar: reviewed BOT bank-holiday transcriptions (versioned add/remove + known_from), validated against actual source bytes
C = CFG['calendar']
ENTRIES = calendar_entries(C)  # raises on malformed/duplicate/ambiguous reviewed records
EXT_FOR = {'pdf': ('.pdf', pdf_validator), 'png': ('.png', png_validator), 'html': ('.html', nonempty_validator)}
cal_sources, src_review = [], {}
for s in C['sources']:
    ext, validator = EXT_FOR[s['kind']]
    entry_dates = [r['date'] for r in ENTRIES if r['source_id'] == s['id']]
    st = {'source_id': s['id'], 'kind': s['kind'], 'role': s['role'], 'years': ','.join(map(str, s['years'])), 'url': s['url'],
          'parent_url': s.get('parent_url'), 'known_from': s.get('known_from'), 'circular_date': s.get('circular_date'),
          'gazette_date': s.get('gazette_date'), 'verified_by': s.get('verified_by'), 'page_ref': s.get('page_ref'),
          'source_quality': s.get('source_quality'), 'n_entries': len(entry_dates), 'notes': s.get('notes')}
    pre = None
    if s.get('preflight_file') and (PREFLIGHT / s['preflight_file']).is_file():
        pre = register_preflight('preflight:' + s['id'], s['url'], PREFLIGHT / s['preflight_file'], REG,
                                 'codex_preflight_download_visually_reviewed_original_time_unknown')
    net = exact_asset_net(NET, s['url'], C['exact_embedded_assets']) if s['kind'] == 'png' else NET
    e = try_fetch(s['id'], s['url'], f"raw/calendar/{s['id']}{ext}", EXT, REG, net, BUDGET, refresh=REFRESH, offline=OFFLINE,
                  validate=validator, max_bytes=C['max_source_bytes'], meta={'source_group': 'calendar', 'parent_url': s.get('parent_url')})
    save_reg()
    live_sha = e.get('sha256')
    if live_sha and s['kind'] == 'png' and urlparse(e.get('resolved_url') or s['url']).path != urlparse(s['url']).path:
        st['live_note'] = f"resolved to a different asset path {e.get('resolved_url')}; live bytes not used"
        live_sha = None
    st.update(live_status=e.get('last_status'), live_sha256=live_sha, live_error=e.get('error'),
              reviewed_copy=s.get('preflight_file'), reviewed_copy_sha256=(pre or {}).get('sha256'))
    if s['kind'] == 'pdf' and (pre or live_sha):
        try:  # supplementary only: image-only PDFs have no text layer, which is NOT an empty calendar
            txt, npages = pdf_text((PREFLIGHT / s['preflight_file']).read_bytes() if pre else (EXT / e['path']).read_bytes())
            st.update(pages=npages, text_chars=len(txt.strip()),
                      text_layer='none_image_only_visual_review_used' if not txt.strip() else 'present_not_used_for_records')
        except Exception as ex:
            st['text_layer'] = f'pypdf_failed: {type(ex).__name__}: {str(ex)[:120]}'
    text_dates = set()
    if s['kind'] == 'html' and live_sha:
        html = (EXT / e['path']).read_bytes().decode('utf-8', errors='replace')
        txt = html_to_text(html)
        text_dates = {m['date'] for y in s['years'] for m in extract_thai_dates(txt, y)}
        if s.get('embedded_asset_check'):
            st['embedded_asset_reference_found'] = s['embedded_asset_check'] in html
    ok, why = review_source_status(s, (pre or {}).get('sha256'), live_sha, text_dates, entry_dates)
    if ok and s.get('parent_source_id'):
        parent = src_review.get(s['parent_source_id']) or {}
        if not parent.get('live_sha256'):
            ok, why = False, 'parent_page_snapshot_unavailable'
        elif parent.get('embedded_asset_reference_found') is False:
            GAPS.append(f"calendar: parent page {s['parent_source_id']} snapshot does not contain the asset reference "
                        f"(may be script-loaded); {s['id']} applied on Codex visual review of the embed")
    st.update(review_ok=bool(ok and entry_dates), review_status=why if entry_dates else f'no_entries ({why})')
    src_review[s['id']] = st
    cal_sources.append(st)
    if entry_dates and not ok:
        GAPS.append(f"calendar: {s['id']} reviewed entries NOT applied ({why})")
    print(f"  {s['id']} [{s['kind']}/{s['role']}]: live={st['live_status']} reviewed_copy={'yes' if pre else 'no'} -> {st['review_status']}", flush=True)

src_by_id = {s['id']: s for s in C['sources']}
recs = []
for r in ENTRIES:
    s, st = src_by_id[r['source_id']], src_review[r['source_id']]
    recs.append({**r, 'scope': C['scope'], 'is_national_public_holiday': None, 'source_kind': s['kind'], 'source_role': s['role'],
                 'source_url': s['url'], 'parent_url': s.get('parent_url'), 'available_from_local': available_from_local(r['known_from']).isoformat(),
                 'verified_by': s.get('verified_by'), 'source_review_status': st['review_status'], 'applied': st['review_ok'],
                 'exclusion_reason': None if st['review_ok'] else st['review_status'],
                 'reviewed_copy_sha256': st.get('reviewed_copy_sha256'), 'live_sha256': st.get('live_sha256')})
CAL_RECORDS = pd.DataFrame(recs, columns=CAL_RECORD_COLUMNS).sort_values(['date', 'known_from', 'source_id']).reset_index(drop=True)
CAL_SOURCES = pd.DataFrame(cal_sources)
views = []
for v in C['as_of_views']:
    t = dt.datetime.fromisoformat(v)
    days = bank_holidays_as_of(CAL_RECORDS, t)
    for y in C['years']:
        yd = sorted(d for d in days if d.startswith(str(y)))
        views.append({'as_of_local': v, 'year': y, 'coverage_status': calendar_year_coverage(CAL_RECORDS, C['sources'], y, t),
                      'completeness_claim': 'not_claimed', 'n_bank_holidays_known': len(yd), 'dates': ';'.join(yd)})
CAL_VIEWS = pd.DataFrame(views)
FEST_EVIDENCE = festival_rule_evidence(CAL_RECORDS, C['recurring_fixed_festivals'], C['years'])
write_csv(REP / 'calendar_sources.csv', CAL_SOURCES)
write_csv(REP / 'calendar_records.csv', CAL_RECORDS)
write_csv(REP / 'calendar_as_of_views.csv', CAL_VIEWS)
write_csv(REP / 'calendar_festival_evidence.csv', FEST_EVIDENCE)
for st in cal_sources:
    COVERAGE.append({'source_group': 'calendar', 'resource_id': st['source_id'], 'year': st['years'], 'format': st['kind'],
                     'rows': st['n_entries'], 'n_columns': None, 'mapped_fields': None, 'missing_fields': None, 'unmapped_headers': None,
                     'coverage_start': None, 'coverage_end': None, 'status': f"{st['live_status']};{st['review_status']}",
                     'sha256': st.get('reviewed_copy_sha256') or st.get('live_sha256'), 'note': st.get('source_quality') or st.get('notes')})
GAPS.extend('calendar: ' + g for g in C['known_gaps'])
n_app = int(CAL_RECORDS['applied'].sum())
bases_ok = sorted({y for s in C['sources'] if s['role'] == 'annual_base' and src_review[s['id']]['review_ok'] for y in s['years']})
CAL_STATUS = {'summary': (f"reviewed BOT bank-holiday records applied {n_app}/{len(CAL_RECORDS)}; annual bases applied for years {bases_ok} "
                          f"of {C['years']}; sources with bytes/snapshot {sum(1 for s in cal_sources if s.get('live_sha256') or s.get('reviewed_copy_sha256'))}"
                          f"/{len(cal_sources)}; scope {C['scope']} (not public holidays); completeness not claimed"),
              'records_applied': n_app, 'records_total': len(CAL_RECORDS), 'annual_bases_applied_years': bases_ok}
RUN['sections']['calendar'] = {'sources': cal_sources, **CAL_STATUS}
print(CAL_STATUS['summary'])
CAL_VIEWS[['as_of_local', 'year', 'coverage_status', 'n_bank_holidays_known']]

  bot_holiday_main_page [html/context]: live=cache_hit_hash_verified reviewed_copy=no -> no_entries (html_snapshot_date_text_found)


  bot_news_20210106_1 [html/parent_page]: live=cache_hit_hash_verified reviewed_copy=no -> no_entries (html_snapshot_date_text_found)


  bot_holidays_2021_png [png/annual_base]: live=cache_hit_hash_verified reviewed_copy=yes -> live_bytes_identical_to_reviewed_copy


Ignoring wrong pointing object 16 0 (offset 0)


Ignoring wrong pointing object 150 0 (offset 0)


  bot_holidays_2022_pdf [pdf/annual_base]: live=cache_hit_hash_verified reviewed_copy=yes -> live_bytes_identical_to_reviewed_copy


... (9 outputs left out)

  bot_holidays_2025_pdf [pdf/annual_base]: live=cache_hit_hash_verified reviewed_copy=yes -> live_bytes_identical_to_reviewed_copy


  bot_news_20241127 [html/special_announcement]: live=cache_hit_hash_verified reviewed_copy=no -> html_snapshot_date_text_found


reviewed BOT bank-holiday records applied 95/95; annual bases applied for years [2021, 2022, 2023, 2024, 2025] of [2021, 2022, 2023, 2024, 2025]; sources with bytes/snapshot 13/13; scope BOT_national_financial_institution_holiday (not public holidays); completeness not claimed


,as_of_local,year,coverage_status,n_bank_holidays_known
0,2021-01-06T18:00,2021,unknown_annual_source_not_yet_known,0
1,2021-01-06T18:00,2022,unknown_annual_source_not_yet_known,0
2,2021-01-06T18:00,2023,unknown_annual_source_not_yet_known,0
3,2021-01-06T18:00,2024,unknown_annual_source_not_yet_known,0
4,2021-01-06T18:00,2025,unknown_annual_source_not_yet_known,0
5,2021-01-07T18:00,2021,reviewed_annual_base_known_later_amendments_un...,20
6,2021-01-07T18:00,2022,unknown_annual_source_not_yet_known,0
7,2021-01-07T18:00,2023,unknown_annual_source_not_yet_known,0
8,2021-01-07T18:00,2024,unknown_annual_source_not_yet_known,0
9,2021-01-07T18:00,2025,unknown_annual_source_not_yet_known,0


In [10]:
# G. Feature-source readiness, schema coverage, Thai audit report and sources manifest
def _status(n_ok, n_total):
    return 'blocked' if n_ok == 0 else ('collected' if n_ok == n_total else 'partial')


traffic_ok = int((TRAFFIC_META['normalize_status'] == 'normalized').sum()) if len(TRAFFIC_META) else 0
n_years_cal = len(CFG['calendar']['years'])
evidence = {
    'holiday_festival': ('partial' if CAL_STATUS['records_applied'] else 'blocked',
                         f"{CAL_STATUS['summary']}; recurring Songkran/New Year rules are deterministic; no national public-holiday calendar"),
    'weather': (W_STATUS['readiness_status'], W_STATUS['summary']),
    'season': ('derived', 'calendar-month proxy computed from the target date in the next task; no external source'),
    'timeblock': ('derived', 'province x 6 h target blocks derived from each event\'s own time in the next task (label side only)'),
    'traffic': (_status(traffic_ok, len(CFG['traffic']['survey_years_be'])),
                f'normalized DOH AADT survey years: {traffic_ok}/{len(CFG["traffic"]["survey_years_be"])}; availability proxy max(created,last_modified)+1d'),
    'history': ('blocked', 'report availability at issue time not verified; conditional research only'),
}
READINESS = pd.DataFrame([{'group': r['group'], 'status': evidence[r['group']][0], 'evidence': evidence[r['group']][1],
                           **{k: r[k] for k in ('next_stage_allowed_use', 'granularity', 'available_at_policy', 'leakage_risk')}}
                          for r in CFG['readiness']])
assert set(READINESS['group']) == {'holiday_festival', 'weather', 'season', 'timeblock', 'traffic', 'history'}
COVERAGE_DF = pd.DataFrame(COVERAGE)
write_csv(REP / 'feature_source_readiness.csv', READINESS)
write_csv(REP / 'external_schema_coverage.csv', COVERAGE_DF)

md = ['# รายงานตรวจแหล่งข้อมูลภายนอก v2 (Notebook 09)', '',
      f'สร้างเมื่อ (UTC): {utc_now()} | สถานะ: งานวิจัยระหว่างดำเนินการ ไม่ใช่ระบบพยากรณ์ที่พร้อมใช้ | ยังไม่มีการเทียบโมเดล', '',
      '## 1. หลักฐานวันที่รายปี (ACC_CODE เดิมตรงตัว + วันที่ typed ใน XLSX + บริบทจังหวัด/เวลา)', '',
      '| ปี ค.ศ. | แถว CSV v1 | กำกวมใน v1 | resolve (ID+วันที่+บริบท) | กำกวมที่ resolve | ไม่ตรง | M/D ตรง/เทียบ | D/M ตรง/เทียบ | บริบท | นโยบาย |',
      '|---|---|---|---|---|---|---|---|---|---|']
for r in DATE_SUMMARY:
    md.append(f"| {r['year_ce']} | {r['csv_rows']} | {r['csv_ambiguous_slash_rows']} | {r.get('n_resolved', '')} | {r.get('n_ambiguous_resolved', '')} | "
              f"{r.get('n_mismatch', '')} | {r.get('rule_mdy_agree', '')}/{r.get('rule_compared', '')} | {r.get('rule_dmy_agree', '')}/{r.get('rule_compared', '')} | "
              f"{r.get('context_counts', '')} | {r['use_policy']} |")
md += ['', 'ข้อสรุปเป็นรายปีและรายแถว ไม่ใช้ parser M/D หรือ D/M แบบรวมทั้งปี (2022 มีรูปแบบผสม); การ resolve เป็นการอ้างอิงคู่ข้อมูลของผู้เผยแพร่ '
       'ไม่ใช่ ground truth อิสระ; ID ซ้ำ/ขัดแย้ง/บริบทไม่ตรงยังถูกกักกัน; แถวที่มีเฉพาะใน XLSX ไม่ถูกเพิ่ม; ปี 2026 ไม่มี ID จึงเป็น diagnostic เท่านั้น',
       f'เอกสาร data dictionary ของ MOT: {dd_status}', '',
       '## 2. ความพร้อมของแหล่ง feature', '', '| กลุ่ม | สถานะ | หลักฐาน | การใช้ที่อนุญาตในขั้นถัดไป | นโยบาย available-at | ความเสี่ยง leakage |', '|---|---|---|---|---|---|']
for r in READINESS.itertuples():
    md.append(f'| {r.group} | {r.status} | {r.evidence} | {r.next_stage_allowed_use} | {r.available_at_policy} | {r.leakage_risk} |')
md += ['', '## 3. จราจร (DOH AADT)', '', CFG['traffic']['unit_note'], CFG['traffic']['availability_policy'], '']
if len(TRAFFIC_META):
    md += ['แถวที่ว่างทุกคอลัมน์ใน CSV เป็นบรรทัดว่าง (non-record) ไม่ใช่จุดสำรวจซ้ำ จึงนับแยกและไม่เข้าสถิติ; key ซ้ำนับเฉพาะแถวที่มีข้อมูลและ key ครบ; '
           'แถวที่ระบุตัวตนไม่ครบไม่ถูกรวม/เดา; ชื่อจังหวัดใช้แผนที่ alias ที่ตรวจแล้วแบบตรงตัวเท่านั้น '
           f"({'; '.join(f'{k} -> {v}' for k, v in CFG['province_aliases']['map'].items())}) โดยเก็บข้อความดิบไว้ใน province_raw", '',
           '| ปีสำรวจ พ.ศ. | สถานะ | แถวที่อ่าน | บรรทัดว่าง | station | key ซ้ำ | ระบุตัวตนไม่ครบ | total ใช้ได้ (station) | total ไม่ตรงผลรวม | '
           'available_from (proxy, UTC) | ฐาน |', '|---|---|---|---|---|---|---|---|---|---|---|']
    for r in TRAFFIC_META.to_dict('records'):
        md.append(f"| {r['survey_year_be']} | {r['normalize_status']} | {r.get('rows_parsed')} | {r.get('rows_blank_nonrecord')} | {r.get('rows_station')} | "
                  f"{r.get('rows_duplicate_key')} | {r.get('rows_incomplete_identity')} | {r.get('station_valid_total_rows')} | {r.get('total_mismatch_rows')} | "
                  f"{r.get('available_from_proxy_utc')} | {r.get('available_from_basis')} |")
md += ['', '## 4. ภูมิศาสตร์และจุดสอบถามอากาศ', '', CFG['geography']['source_quality_label'], CFG['geography']['point_meaning'], GEO_STATUS['summary'], '',
       '## 5. พยากรณ์อากาศย้อนหลัง (Open-Meteo Previous Runs, JMA GSM)', '', W_STATUS['summary'], CFG['weather']['lead_note'], CFG['weather']['aggregation_note'], '',
       '## 6. ปฏิทินวันหยุดธนาคาร (ธปท.) และเทศกาล', '', CFG['calendar']['scope_note'], CFG['calendar']['known_from_convention'], CAL_STATUS['summary'], '',
       '| แหล่ง | ชนิด | บทบาท | known_from | สถานะตรวจ |', '|---|---|---|---|---|']
md += [f"| {s['source_id']} | {s['kind']} | {s['role']} | {s.get('known_from')} | {s['review_status']} |" for s in cal_sources]
md += ['', '## 7. ช่องว่างของแหล่งข้อมูล (บันทึกจากการรันจริง)', '']
md += [f'- {g}' for g in GAPS] or ['- ไม่มี']
(REP / 'source_audit_th.md').write_text('\n'.join(md) + '\n', 'utf-8')

fn_src = functions_cells_source(BACKEND / 'notebooks' / '09_external_collection_date_evidence.ipynb')
versions = {'python': sys.version.split()[0], 'pandas': pd.__version__, 'numpy': np.__version__}
for mod in ('openpyxl', 'pypdf', 'shapely', 'pyarrow'):
    try:
        versions[mod] = __import__(mod).__version__
    except Exception:
        versions[mod] = None
cached = []
for rid, e in sorted(REG['resources'].items()):
    cached.append({'resource_id': rid, 'path': e.get('path'), 'preflight_path': e.get('preflight_path'), 'sha256': e.get('sha256'),
                   'bytes': e.get('bytes'), 'last_status': e.get('last_status'), 'url': e.get('url'), 'provenance': e.get('provenance')})
for rel in DERIVED:
    cached.append({'resource_id': 'derived:' + rel, 'path': rel, 'sha256': sha256_file(EXT / rel), 'bytes': (EXT / rel).stat().st_size,
                   'last_status': 'derived_by_notebook09', 'provenance': 'derived_by_notebook09'})
outputs = [{'path': p.relative_to(OUT).as_posix(), 'sha256': sha256_file(p), 'bytes': p.stat().st_size}
           for p in sorted(REP.iterdir()) if p.is_file() and p.name != 'sources_manifest.json']
MANIFEST = {
    'producer': '09_external_collection_date_evidence', 'schema': 'roadaware-v2-sources-manifest/2',
    'status': 'completed' if not GAPS else 'completed_with_gaps',
    'status_note': 'collection/audit only; no feature panel, no model fit, no claim of beating baselines; passing checks is not acceptance',
    'started_utc': RUN['started_utc'], 'finished_utc': utc_now(), 'offline': OFFLINE, 'refresh': REFRESH,
    'functions_cells_sha256': sha256_bytes(fn_src.encode('utf-8')), 'config_sha256': sha256_file(CFG_PATH),
    'config_path': 'config/external_sources_v2.json', 'external_root': str(EXT), 'output_root': str(OUT), 'v1_data_root': str(V1),
    'preflight_dir': str(PREFLIGHT),
    'v1_raw_manifest_sha256': sha256_file(V1 / 'data' / 'raw' / 'accidents' / 'source_manifest.json'),
    'v1_raw_snapshot_sha256': V1_HASHES_BEFORE, 'versions': versions,
    'network': {'requests_used': BUDGET.used, 'max_requests': BUDGET.max_requests, 'weather_requests_used': W_STATUS['requests_used']},
    'sections': RUN['sections'], 'gaps': GAPS, 'cached_artifacts': cached, 'outputs': outputs,
    'next_stage_interface': {
        'date_rows': 'derived/date_resolution_by_id.csv: use resolution_status == resolved only (ID + typed date + context agreement)',
        'weather_daily': ('derived/weather_daily_jma_gsm.csv.gz and weather_6h_jma_gsm.csv.gz: rain_missing/temp_missing False and issue_bound_ok True; '
                          'rain is a preceding-hour sum bucketed by interval start (rain_interval_convention); negative precipitation hours are masked '
                          'to NaN per hour (n_rain_value_masked, rain_missing_reason, reports/v2/sources/weather_value_quality.csv) - audit only, '
                          'never targets or predictors'),
        'traffic': ('reports/v2/sources/traffic_province_year_summary.csv (join on province_key = v1 panel province) + traffic_resource_metadata.csv '
                    'available_from_proxy_utc; survey_year < target year; station rows in derived/traffic_aadt_normalized.csv with row_class == station '
                    'and total_valid/heavy_share_valid; schema in sections.traffic.station_schema'),
        'calendar': 'reports/v2/sources/calendar_records.csv applied rows replayed with bank_holidays_as_of; coverage via calendar_year_coverage; unknown != ordinary day',
        'points': ('reports/v2/sources/province_points.csv (province_code, name_th = original table text, province_key = v1 panel join key via the '
                   'exact audited alias map, lat, lon)')},
}
write_json(REP / 'sources_manifest.json', MANIFEST)
print(READINESS[['group', 'status', 'evidence']].to_string(index=False))
print('reports ->', REP)

           group    status                                                                                                                                                                                                                                                                                                                                                                        evidence
holiday_festival   partial reviewed BOT bank-holiday records applied 95/95; annual bases applied for years [2021, 2022, 2023, 2024, 2025] of [2021, 2022, 2023, 2024, 2025]; sources with bytes/snapshot 13/13; scope BOT_national_financial_institution_holiday (not public holidays); completeness not claimed; recurring Songkran/New Year rules are deterministic; no national public-holiday calendar
         weather   partial                          JMA GSM previous_day2 chunks with data: 42/80; province-years retained: 204/385, complete (all hours valid): 172/385; negative precipitation hours masked to N

In [11]:
# Final checks (passing them means the bookkeeping is consistent; it does NOT prove the external data are complete or correct)
def _reject_constant(c):
    raise ValueError('non-finite JSON constant ' + c)


assert verify_v1_snapshot(V1, V1MAN) == V1_HASHES_BEFORE, 'v1 raw snapshot changed during notebook 09'
_v1_after = protected_tree_hashes(V1_PROTECTED, V1)
assert _v1_after == V1_TREE_BEFORE, ('protected v1 files changed during notebook 09: '
                                     f'{sorted(k for k in set(_v1_after) | set(V1_TREE_BEFORE) if _v1_after.get(k) != V1_TREE_BEFORE.get(k))[:20]}')
if len(TRAFFIC_NORM):
    assert not TRAFFIC_NORM['row_class'].eq('blank_nonrecord').any(), 'blank CSV non-records must not be station data'
    assert set(TRAFFIC_NORM['row_class']) <= set(AADT_ROW_CLASSES)
for _be, _a in TRAFFIC_ROW_AUDIT.items():
    assert _a['rows_parsed'] == sum(_a[f'rows_{c}'] for c in AADT_ROW_CLASSES), f'traffic row audit does not add up for BE {_be}'
if len(TRAFFIC_PROVINCE):
    assert TRAFFIC_PROVINCE['mean_heavy_share_pct'].dropna().between(0, 100).all() and (TRAFFIC_PROVINCE['median_total_aadt'].dropna() >= 0).all()
if len(POINTS):
    assert POINTS['in_v1_panel'].eq(POINTS['province_key_status'].isin(['v1_panel_exact', 'v1_panel_audited_alias'])).all()
for p in sorted(REP.iterdir()):
    if p.is_file():
        assert p.stat().st_size < 5 * 1024 * 1024, f'report file too large for reports/ (row-level data belongs under data/): {p}'
        if p.suffix == '.json':
            json.loads(p.read_text('utf-8'), parse_constant=_reject_constant)
for o in MANIFEST['outputs']:
    assert sha256_file(OUT / o['path']) == o['sha256'], 'output changed after manifest: ' + o['path']
for a in MANIFEST['cached_artifacts']:
    if a.get('sha256') and a.get('path'):
        assert sha256_file(EXT / a['path']) == a['sha256'], 'cached artifact changed: ' + a['path']
if len(DATE_ROWS):
    ok = DATE_ROWS['resolution_status'] == 'resolved'
    assert DATE_ROWS.loc[ok, 'resolved_date'].notna().all() and DATE_ROWS.loc[~ok, 'resolved_date'].isna().all()
    assert DATE_ROWS.loc[ok, 'context_status'].eq('agree').all(), 'resolved row without context agreement'
    assert not DATE_ROWS['source_year_ce'].isin(CFG['date_evidence']['diagnostic_only_years_ce']).any()
if len(POINTS):
    assert POINTS.loc[POINTS['province_code'].notna(), 'point_within_polygon'].astype(bool).all(), 'query point outside its province'
for agg, n in ((W_DAILY, 24), (W_6H, 6)):
    if len(agg):
        assert agg.loc[agg['rain_sum'].notna(), 'n_valid_rain'].eq(n).all(), 'rain sum from incomplete hours'
        assert agg.loc[agg['temp_mean'].notna(), 'n_valid_temp'].eq(n).all(), 'temperature mean from incomplete hours'
        assert agg['issue_bound_ok'].all(), 'forecast hour whose nominal issue bound + buffer is after the issue time'
        assert agg.loc[agg['n_rain_value_masked'] > 0, 'rain_sum'].isna().all(), 'rain window containing a masked hour was summed'
if len(W_HOURLY):
    _rq = W_HOURLY[CFG['weather']['rain_var'] + '_quality']
    assert not (W_HOURLY[CFG['weather']['rain_var']] < 0).any(), 'negative precipitation left in the normalized hourly table'
    assert W_HOURLY.loc[_rq.eq('negative_value_masked'), CFG['weather']['rain_var']].isna().all()
    assert int(_rq.eq('negative_value_masked').sum()) == len(W_QFLAGS) == int(W_QUALITY['n_negative_masked'].sum()), 'masked-hour lineage does not add up'
    assert not W_HOURLY[CFG['weather']['temp_var'] + '_quality'].eq('negative_value_masked').any(), 'temperature must never be masked as negative'
    assert (W_QFLAGS['raw_value'] < 0).all(), 'lineage must keep the raw negative value'
if len(TRAFFIC_PROVINCE):
    assert not any('sum' in c for c in TRAFFIC_PROVINCE.columns), 'traffic summaries must not sum survey points'
if len(CAL_RECORDS):
    assert CAL_RECORDS['scope'].eq(CFG['calendar']['scope']).all()
    assert not CAL_RECORDS['is_national_public_holiday'].eq(True).any(), 'bank holidays must not be labelled national public holidays'
    app = CAL_RECORDS[CAL_RECORDS['applied'].astype(bool)]
    assert not (app['live_sha256'].notna() & app['reviewed_copy_sha256'].notna() & (app['live_sha256'] != app['reviewed_copy_sha256'])).any(), \
        'applied reviewed record whose live bytes differ from the reviewed copy'
    assert app['source_review_status'].isin(['live_bytes_identical_to_reviewed_copy', 'reviewed_copy_only_live_unavailable',
                                             'live_bytes_match_pinned_reviewed_sha256', 'html_snapshot_date_text_found']).all()
    amend = app[app['source_role'] == 'amendment']
    if len(amend):  # as-of replay must honour the year-end amendment (remove 2024-01-02, add 2023-12-29) only after its known_from
        final = bank_holidays_as_of(CAL_RECORDS, dt.datetime(2026, 1, 1))
        assert '2024-01-02' not in final and '2023-12-29' in final, 'amendment not applied in the final as-of view'
        kf = available_from_local(amend['known_from'].iloc[0])
        assert '2023-12-29' not in bank_holidays_as_of(CAL_RECORDS, kf - dt.timedelta(seconds=1)), 'amendment used before known_from'
print('final checks passed | status:', MANIFEST['status'], '| protected v1 files unchanged:', len(V1_TREE_BEFORE))
print('network requests used:', MANIFEST['network']['requests_used'], '| gaps:', len(MANIFEST['gaps']))
for g in MANIFEST['gaps']:
    print('  gap:', g)

final checks passed | status: completed_with_gaps | protected v1 files unchanged: 135
network requests used: 35 | gaps: 11
  gap: MOT data dictionary: no data-dictionary resource listed in the current MOT package (nothing fetched). Codex reported the old MOT data-dictionary PDF URL returns HTTP 404 (exact URL not recorded in the task files). A cached search snippet claiming MM/DD is NOT a verified document. Any dictionary resource listed in the live package is snapshotted only; its content is not used as date-format proof. Typed XLSX date cells matched by original ACC_CODE are the primary evidence.
  gap: traffic: BE 2567 has 2130 fully blank CSV lines: counted as non-records and excluded from station data/statistics (raw bytes kept; not duplicate survey points)
  gap: traffic: availability uses max(created, last_modified) + 1 day catalog proxy; it does not prove the current bytes were delivered then
  gap: weather coverage incomplete: 172/385 province-years complete, 204 retained (sto

## ข้อจำกัดและสัญญาสำหรับขั้นถัดไป (005: v2 feature panel – งานแยกต่างหาก)

- **วันที่:** ใช้ `data/external_v2/derived/date_resolution_by_id.csv` เฉพาะแถว `resolution_status == 'resolved'` (ID เดิมตรงกัน วันที่ typed ใน XLSX เห็นพ้องกันไม่กำกวม อยู่ในชุดการอ่านของแถวนั้น และจังหวัด/เวลาตรงกัน) พร้อม hash ใน `sources_manifest.json`; `context_conflict`/`context_unverified`/ID ซ้ำขัดแย้ง ยังถูกกักกัน; ไม่ใช้ parser M/D หรือ D/M ทั้งปี; ปี 2026 (ไม่มี ID) เป็น diagnostic เท่านั้น; ห้ามเพิ่มแถวที่มีเฉพาะใน XLSX
- **จราจร:** `reports/v2/sources/traffic_province_year_summary.csv` (mean/median/จำนวนจุดสำรวจ ต่อจังหวัด × ปีสำรวจ ไม่รวมแถว key ซ้ำ) และ `available_from_proxy_utc` ใน `traffic_resource_metadata.csv` = max(created, last_modified) + 1 วัน; ใช้ได้เฉพาะปีสำรวจ < ปีเป้าหมาย และ available_from ≤ เวลาออกพยากรณ์ ห้ามรวมเป็นจำนวนรถ ห้ามเติมค่าย้อน/ล่วงหน้า
- **อากาศ:** `derived/weather_daily_jma_gsm.csv.gz` และ `weather_6h_jma_gsm.csv.gz` ใช้ได้เฉพาะแถว `rain_missing == False`/`temp_missing == False` และ `issue_bound_ok == True`; ค่าที่ขาดคงเป็นค่าว่าง (ไม่เป็น 0); จุดเป็น proxy ของจังหวัด; ความครอบคลุมดู `weather_coverage.csv`; ค่าฝนติดลบ (เช่น -0.1 mm) ถูกทำเป็นค่าว่างเฉพาะชั่วโมงนั้น (ไม่ปัด/ไม่ตัดเป็น 0, JSON ดิบไม่ถูกแก้) และช่วงฝน 6 ชม./วันที่มีชั่วโมงนั้นเป็นค่าว่าง ดู `weather_value_quality.csv`
- **ปฏิทิน/เทศกาล:** เทศกาลวันที่คงที่ใช้ได้ทุกปี; วันหยุดธนาคารใช้เฉพาะแถว `applied == True` ใน `calendar_records.csv` ผ่านฟังก์ชัน `bank_holidays_as_of` ณ เวลาออกพยากรณ์ และต้องตรวจ `calendar_year_coverage` (ไม่ทราบ ≠ ไม่ใช่วันหยุด); ไม่มีธงวันหยุดราชการ
- **ฤดูกาล/ช่วงเวลา:** ฤดูกาลเป็น proxy จากเดือน; ช่วง 6 ชม. เป็นหน่วยของเป้าหมาย ไม่ใช่ feature จากเวลาของเหตุการณ์เดียวกัน
- **ประวัติ (lags):** ถูกบล็อกสำหรับการอ้างใช้งานจริง ใช้ได้เฉพาะ sensitivity แบบมีเงื่อนไขตาม policy
- notebook นี้ **ยังไม่ผ่านการตรวจรับ** จนกว่า Codex จะรันจริงและตรวจ input/output/hash/coverage; ไม่ใช่ระบบพยากรณ์ออนไลน์ และไม่มีการเทียบโมเดลที่นี่